# 03 · Power BI dashboard (Direct Lake) — built as code
Attach **lh_fraud** as default lakehouse, then **Run all** (≈ 3–5 min). Re-runnable: it overwrites the BI tables, the model and the report.

1. `pbi_*` Delta tables shaped for reporting (from the Gold tables of notebook 02)
2. Direct Lake semantic model **sm_fraud_copilot** + DAX measures + relationship
3. Report **Fraud Copilot — Dashboard** (4 pages, custom theme)

No Power BI Copilot is used (capacity is judged).

In [ ]:
%%configure -f
{"defaultLakehouse": {"name": "lh_fraud"}}

In [ ]:
# plain pip (the %pip magic is blocked when the notebook runs as a scheduled/API job)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "semantic-link-labs"], check=True)

In [ ]:
import json
import pandas as pd
from pyspark.sql import functions as F

def save(df, name):
    (df if not isinstance(df, pd.DataFrame) else spark.createDataFrame(df)) \
        .write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(name)

# the queue also holds one non-alert demo row (FAIBLE, avoided false positive) -> not counted as an alert
q = spark.table("gold_alert_queue").filter(F.col("risk_level") != "FAIBLE")
print("queue columns:", q.columns)
print("risk levels:", [r[0] for r in q.select("risk_level").distinct().collect()])

SCEN = {"CARD_TESTING": "Test de carte", "ACCOUNT_TAKEOVER": "Prise de contrôle de compte",
        "CARTE_CLONEE_ETRANGER": "Carte clonée à l'étranger", "SKIMMING_DAB": "Skimming DAB",
        "CARTE_VOLEE_SANS_CONTACT": "Carte volée sans contact", "FRAUDE_EN_LIGNE_DISCRETE": "Fraude en ligne discrète"}
scen_map = F.create_map([F.lit(x) for kv in SCEN.items() for x in kv])
SCEN_EN = {"CARD_TESTING": "Card testing", "ACCOUNT_TAKEOVER": "Account takeover", "CARTE_CLONEE_ETRANGER": "Card cloned abroad",
           "SKIMMING_DAB": "ATM skimming", "CARTE_VOLEE_SANS_CONTACT": "Stolen card, contactless",
           "FRAUDE_EN_LIGNE_DISCRETE": "Discreet online fraud"}
scen_en = F.create_map([F.lit(x) for kv in SCEN_EN.items() for x in kv])

alerts = (q.withColumn("alert_date", F.to_date("timestamp"))
           .withColumn("alert_hour", F.hour("timestamp"))
           .withColumn("alert_month", F.date_format("timestamp", "yyyy-MM"))
           .withColumn("periode", F.when(F.col("period") == "test", "Flux live (août–sept.)").otherwise("Validation (juin–juil.)"))
           .withColumn("verite", F.when(F.col("period") == "test", "Inconnue (flux live)")
                                  .when(F.col("is_fraud") == 1, "Fraude confirmée").otherwise("Légitime"))
           .withColumn("vu_par_regles", F.when(F.col("legacy_alert") == 1, "Aussi alerté par les règles")
                                         .otherwise("Manqué par les règles"))
           .withColumn("typologie", F.coalesce(scen_map[F.col("fraud_scenario")], F.col("fraud_scenario"), F.lit("—")))
           # English labels for the EN pages
           .withColumn("periode_en", F.when(F.col("period") == "test", "Live flow (Aug–Sep)").otherwise("Validation (Jun–Jul)"))
           .withColumn("verite_en", F.when(F.col("period") == "test", "Unknown (live flow)")
                                     .when(F.col("is_fraud") == 1, "Confirmed fraud").otherwise("Legitimate"))
           .withColumn("vu_par_regles_en", F.when(F.col("legacy_alert") == 1, "Also flagged by rules").otherwise("Missed by rules"))
           .withColumn("typologie_en", F.coalesce(scen_en[F.col("fraud_scenario")], F.col("fraud_scenario"), F.lit("—")))
           .withColumn("risk_en", F.when(F.col("risk_level") == "ELEVE", "HIGH").when(F.col("risk_level") == "MOYEN", "MEDIUM")
                                   .when(F.col("risk_level") == "FAIBLE", "LOW").otherwise(F.col("risk_level")))
           .withColumn("bande_score", F.when(F.col("score_model") >= 0.9, "1 · ≥ 0,90").when(F.col("score_model") >= 0.7, "2 · 0,70–0,90")
                                       .when(F.col("score_model") >= 0.5, "3 · 0,50–0,70").otherwise("4 · < 0,50"))
           .withColumn("is_fraud_n", F.coalesce(F.col("is_fraud").cast("int"), F.lit(0)))
           .withColumn("amount_tnd", F.col("amount_tnd").cast("double")))
save(alerts, "pbi_alerts")

ex = spark.table("gold_alert_explanations")
factors = (ex.withColumn("factor_kind", F.when(F.col("source").startswith("rules"), "Règle métier")
                                        .when(F.lower(F.col("source")).contains("shap"), "Modèle (SHAP)")
                                        .otherwise(F.col("source")))
             .withColumn("factor_kind_en", F.when(F.col("factor_kind") == "Règle métier", "Business rule")
                                            .when(F.col("factor_kind") == "Modèle (SHAP)", "Model (SHAP)").otherwise(F.col("factor_kind")))
             .select("transaction_id", "factor", "factor_kind", "factor_kind_en", "points", "observed", "usual"))
save(factors, "pbi_factors")
print("factor kinds:", [r[0] for r in factors.select("factor_kind").distinct().collect()])

In [ ]:
m = {r["metric"]: float(r["value"]) for r in spark.table("gold_model_metrics").collect()}
g = lambda k, d=0.0: m.get(k, d)
vr, vf = g("valid_rows"), g("valid_frauds")
# exact top-1 % counts from the validation alerts themselves (no rounding drift vs the app / STATUS)
# exact validation counts from gold_scores x silver labels (the alert queue also holds a non-alert demo row,
# and gold_model_metrics stores frauds_caught_legacy = 116 while the data says 117)
_lab = spark.table("silver_transactions").select("transaction_id", "is_fraud")
_ex = (spark.table("gold_scores").filter(F.col("period") == "valid").join(_lab, "transaction_id")
       .agg(F.sum(F.col("is_alert").cast("int")).alias("a"),
            F.sum(F.when((F.col("is_alert") == 1) & (F.col("is_fraud") == 1), 1).otherwise(0)).alias("tp"),
            F.sum(F.col("legacy_alert").cast("int")).alias("la"),
            F.sum(F.when((F.col("legacy_alert") == 1) & (F.col("is_fraud") == 1), 1).otherwise(0)).alias("ltp")).first())
top1_alerts, top1_frauds = int(_ex["a"]), int(_ex["tp"])
leg_alerts, leg_tp = int(_ex["la"]), int(_ex["ltp"])
print("exact validation counts:", dict(top1_alerts=top1_alerts, top1_frauds=top1_frauds, leg_alerts=leg_alerts, leg_tp=leg_tp))
top05_alerts, top05_frauds = round(vr * 0.005), round(g("model_top0.5pct_recall") * vf)

compare = pd.DataFrame([
    ("1 · Moteur de règles actuel", leg_alerts, leg_tp, leg_tp / vf, leg_tp / leg_alerts),
    ("2 · Modèle, même volume d'alertes", g("legacy_alerts"), g("frauds_caught_model_same_budget"),
     g("model_same_budget_as_legacy_recall"), g("model_same_budget_as_legacy_precision")),
    ("3 · Modèle, top 1 % (capacité analystes)", top1_alerts, top1_frauds, top1_frauds / vf, top1_frauds / top1_alerts),
    ("4 · Modèle, top 0,5 %", top05_alerts, top05_frauds, g("model_top0.5pct_recall"), g("model_top0.5pct_precision")),
], columns=["configuration", "alerts", "frauds", "recall", "precision"])
compare["false_alerts"] = compare.alerts - compare.frauds
compare["configuration_en"] = ["1 · Current rule engine", "2 · Model, same alert volume",
                               "3 · Model, top 1% (analyst capacity)", "4 · Model, top 0.5%"]
save(compare, "pbi_compare")

kpi = pd.DataFrame([{
    "frauds_model": g("frauds_caught_model_same_budget"), "frauds_rules": leg_tp,
    "recall_model": g("model_same_budget_as_legacy_recall"), "recall_rules": leg_tp / vf,
    "precision_top1": top1_frauds / top1_alerts, "precision_rules": leg_tp / leg_alerts,
    "false_alerts_avoided": (leg_alerts - leg_tp) - (top1_alerts - top1_frauds),
    "extra_frauds": top1_frauds - leg_tp,
    "pr_auc_model": g("pr_auc_model"), "pr_auc_rules": g("pr_auc_rules"),
    "valid_frauds": vf, "valid_rows": vr}])
save(kpi, "pbi_kpi")
print(compare.to_string()); print(kpi.T)

rt = spark.table("gold_model_recall_by_type").toPandas()
rt["typologie"] = rt.fraud_scenario.map(SCEN).fillna(rt.fraud_scenario)
rt["typologie_en"] = rt.fraud_scenario.map(SCEN_EN).fillna(rt.fraud_scenario)
save(rt, "pbi_recall_wide")
long = pd.concat([rt.assign(moteur="Modèle", moteur_en="Model", recall=rt.recall_model, caught=rt.caught_top1pct),
                  rt.assign(moteur="Règles actuelles", moteur_en="Current rules", recall=rt.recall_legacy, caught=rt.caught_legacy)])
save(long[["typologie", "typologie_en", "moteur", "moteur_en", "recall", "caught", "frauds"]], "pbi_recall_type")

save(spark.table("gold_model_leaderboard"), "pbi_leaderboard")
LABEL = {"f_speed_kmh": "Vitesse de déplacement", "f_amount": "Montant", "merchant_category": "Catégorie marchande",
         "channel": "Canal", "f_secs_since_prev": "Temps depuis l'opération précédente", "f_new_merchant": "Nouveau commerçant",
         "transaction_type": "Type d'opération", "entry_mode": "Mode de saisie", "f_n_before": "Ancienneté (nb opérations)",
         "f_key_contactless_24h": "Sans contact sur 24 h", "f_is_foreign": "Opération à l'étranger",
         "merchant_type": "Type de commerçant", "f_same_country_7d": "Opérations dans ce pays (7 j)",
         "f_dist_prev_km": "Distance depuis l'opération précédente", "f_amount_z": "Écart au montant habituel (z)",
         "f_amount_over_limit": "Montant / plafond", "f_amount_ratio": "Montant / moyenne habituelle",
         "f_merchant_frauds_30d": "Commerçant lié à des fraudes (30 j)", "f_night_share_before": "Habitude d'opérer la nuit",
         "f_key_new_merchants_24h": "Nouveaux commerçants sur 24 h"}
imp = spark.table("gold_model_importance").toPandas()
imp["variable"] = imp.feature.map(LABEL).fillna(imp.feature)
LABEL_EN = {"f_speed_kmh": "Travel speed", "f_amount": "Amount", "merchant_category": "Merchant category", "channel": "Channel",
            "f_secs_since_prev": "Time since previous operation", "f_new_merchant": "New merchant", "transaction_type": "Operation type",
            "entry_mode": "Entry mode", "f_n_before": "History length (nb ops)", "f_key_contactless_24h": "Contactless in 24 h",
            "f_is_foreign": "Operation abroad", "merchant_type": "Merchant type", "f_same_country_7d": "Ops in this country (7 d)",
            "f_dist_prev_km": "Distance from previous op", "f_amount_z": "Amount deviation (z)", "f_amount_over_limit": "Amount / limit",
            "f_amount_ratio": "Amount / usual average", "f_merchant_frauds_30d": "Merchant linked to frauds (30 d)",
            "f_night_share_before": "Habit of night operations", "f_key_new_merchants_24h": "New merchants in 24 h"}
imp["variable_en"] = imp.feature.map(LABEL_EN).fillna(imp.feature)
save(imp, "pbi_importance")
print("BI tables written")

In [ ]:
import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs import directlake

DS, LH = "sm_fraud_copilot", "lh_fraud"
TABLES = ["pbi_alerts", "pbi_factors", "pbi_kpi", "pbi_compare", "pbi_recall_type", "pbi_recall_wide",
          "pbi_leaderboard", "pbi_importance"]
REBUILD_MODEL = False   # True only if the pbi_* table list/columns change (keeps the report bound otherwise)
if REBUILD_MODEL or DS not in set(fabric.list_datasets()["Dataset Name"]):
    # lh_fraud is schema-enabled: tables live under dbo
    directlake.generate_direct_lake_semantic_model(dataset=DS, tables=["dbo." + t for t in TABLES], source=LH,
                                                   source_type="Lakehouse", overwrite=True, refresh=False)
print("model ready:", DS)

In [ ]:
from sempy_labs.tom import connect_semantic_model

MEASURES = {
    "pbi_alerts": [
        ("Alertes", "COUNTROWS(pbi_alerts)", "#,0"),
        ("Montant en jeu (TND)", "SUM(pbi_alerts[amount_tnd])", "#,0"),
        ("Score moyen", "AVERAGE(pbi_alerts[score_model])", "0.000"),
        ("Alertes risque élevé", "CALCULATE([Alertes], pbi_alerts[risk_level] = \"ELEVE\")", "#,0"),
        ("Manquées par les règles", "CALCULATE([Alertes], pbi_alerts[legacy_alert] = 0)", "#,0"),
        ("Fraudes confirmées", "SUM(pbi_alerts[is_fraud_n])", "#,0"),
        ("Précision des alertes (validation)",
         "DIVIDE([Fraudes confirmées], CALCULATE([Alertes], pbi_alerts[period] = \"valid\"))", "0.0%"),
    ],
    "pbi_factors": [("Occurrences", "COUNTROWS(pbi_factors)", "#,0")],
    "pbi_kpi": [
        ("Fraudes détectées · modèle", "SUM(pbi_kpi[frauds_model])", "#,0"),
        ("Fraudes détectées · règles", "SUM(pbi_kpi[frauds_rules])", "#,0"),
        ("Rappel · modèle", "SUM(pbi_kpi[recall_model])", "0.0%"),
        ("Rappel · règles", "SUM(pbi_kpi[recall_rules])", "0.0%"),
        ("Précision top 1 %", "SUM(pbi_kpi[precision_top1])", "0.0%"),
        ("Précision · règles", "SUM(pbi_kpi[precision_rules])", "0.0%"),
        ("Fausses alertes évitées", "SUM(pbi_kpi[false_alerts_avoided])", "#,0"),
        ("Fraudes en plus", "SUM(pbi_kpi[extra_frauds])", "#,0"),
        ("PR-AUC · modèle", "SUM(pbi_kpi[pr_auc_model])", "0.000"),
        ("Gain de rappel (×)", "DIVIDE([Rappel · modèle], [Rappel · règles])", "0.0\"×\""),
    ],
    "pbi_compare": [
        ("Alertes générées", "SUM(pbi_compare[alerts])", "#,0"),
        ("Fraudes détectées", "SUM(pbi_compare[frauds])", "#,0"),
        ("Fausses alertes", "SUM(pbi_compare[false_alerts])", "#,0"),
    ],
    "pbi_recall_type": [("Rappel", "AVERAGE(pbi_recall_type[recall])", "0%")],
    "pbi_leaderboard": [("PR-AUC", "SUM(pbi_leaderboard[pr_auc])", "0.000"),
                        ("Rappel top 1 % (modèles)", "SUM(pbi_leaderboard[top1pct_recall])", "0.0%")],
    "pbi_importance": [("Importance", "SUM(pbi_importance[gain_share])", "0.0%")],
    "pbi_recall_wide": [("Fraudes attrapées (modèle)", "SUM(pbi_recall_wide[caught_top1pct])", "#,0")],
}
PCT_COLS = {"pbi_compare": ["recall", "precision"], "pbi_recall_wide": ["recall_model", "recall_legacy"]}
INT_COLS = {"pbi_compare": ["alerts", "frauds", "false_alerts"],
            "pbi_recall_wide": ["frauds", "caught_top1pct", "caught_legacy"], "pbi_alerts": ["amount_tnd"]}
NUM_FMT = {("pbi_alerts", "score_model"): "0.000"}

EN_COLS = {"pbi_alerts": ["periode_en", "verite_en", "vu_par_regles_en", "typologie_en", "risk_en"],
           "pbi_factors": ["factor_kind_en"], "pbi_compare": ["configuration_en"],
           "pbi_recall_type": ["typologie_en", "moteur_en"], "pbi_recall_wide": ["typologie_en"], "pbi_importance": ["variable_en"]}

with connect_semantic_model(dataset=DS, readonly=False) as tom:
    for t, cols in EN_COLS.items():           # new English label columns -> add them to the Direct Lake tables
        have = {c.Name for c in tom.model.Tables[t].Columns}
        for c in cols:
            if c not in have:
                tom.add_data_column(table_name=t, column_name=c, source_column=c, data_type="String")
    existing = {(m.Parent.Name, m.Name) for m in tom.all_measures()}
    for t, ms in MEASURES.items():
        for name, expr, fmt in ms:
            if (t, name) in existing:
                tom.model.Tables[t].Measures[name].Expression = expr
                tom.model.Tables[t].Measures[name].FormatString = fmt
            else:
                tom.add_measure(table_name=t, measure_name=name, expression=expr, format_string=fmt)
    for t, cols in PCT_COLS.items():
        for c in cols:
            tom.model.Tables[t].Columns[c].FormatString = "0.0%"
    for t, cols in INT_COLS.items():
        for c in cols:
            tom.model.Tables[t].Columns[c].FormatString = "#,0"
    for (t, c), f in NUM_FMT.items():
        tom.model.Tables[t].Columns[c].FormatString = f
    if not any(r.FromTable.Name == "pbi_factors" for r in tom.model.Relationships):
        tom.add_relationship(from_table="pbi_factors", from_column="transaction_id", to_table="pbi_alerts",
                             to_column="transaction_id", from_cardinality="Many", to_cardinality="One")
labs.refresh_semantic_model(dataset=DS)
print("measures + relationship OK")

In [ ]:
import json, uuid

# ---- palette (dark navy)
BG, CARD, BAR, LINE = "#0B1426", "#14213A", "#18284A", "#22324F"
WHITE, LIGHT, MUTED = "#FFFFFF", "#D9E2F2", "#8A9BB8"
BLUE, BLUE2, SLATE, ORANGE, RED, GREEN = "#4C7CF3", "#8FB0FF", "#5B6B8C", "#F59E0B", "#F43F5E", "#22C55E"
DY = 30            # content offset below the taller icon header
PAGE_H = 750

def lit(v):
    return {"expr": {"Literal": {"Value": v}}}

def s(text):
    return lit("'" + str(text).replace("'", "''") + "'")

def color(hex_):
    return {"solid": {"color": s(hex_)}}

def num(v):
    return lit("%sD" % v)

T, F_ = lit("true"), lit("false")
C = lambda t, c: ("Column", t, c)
M = lambda t, m: ("Measure", t, m)

def by_value(table, column, mapping):
    return [{"properties": {"fill": color(c)},
             "selector": {"data": [{"scopeId": {"Comparison": {"ComparisonKind": 0,
                 "Left": {"Column": {"Expression": {"SourceRef": {"Entity": table}}, "Property": column}},
                 "Right": {"Literal": {"Value": "'" + v.replace("'", "''") + "'"}}}}}]}}
            for v, c in mapping.items()]

def by_measure(table, mapping):
    return [{"properties": {"fill": color(c)}, "selector": {"metadata": "%s.%s" % (table, m)}} for m, c in mapping.items()]

def chart_style(fill=BLUE, legend=False, labels=True, extra_points=None):
    return {"categoryAxis": [{"properties": {"labelColor": color(MUTED), "fontSize": num(9), "showAxisTitle": F_, "gridlineShow": F_}}],
            "valueAxis": [{"properties": {"labelColor": color(MUTED), "fontSize": num(9), "showAxisTitle": F_,
                                          "gridlineShow": T, "gridlineColor": color(LINE)}}],
            "legend": [{"properties": {"show": T if legend else F_, "position": s("Top"), "labelColor": color(LIGHT),
                                       "showTitle": F_, "fontSize": num(9)}}],
            "labels": [{"properties": {"show": T if labels else F_, "color": color(LIGHT), "fontSize": num(9),
                                       "labelDisplayUnits": num(1)}}],
            "dataPoint": [{"properties": {"fill": color(fill)}}] + (extra_points or [])}

def _container(x, y, w, h, z, single):
    cfg = {"name": uuid.uuid4().hex[:20],
           "layouts": [{"id": 0, "position": {"x": x, "y": y, "z": z, "width": w, "height": h}}], "singleVisual": single}
    return {"x": x, "y": y, "z": z, "width": w, "height": h, "config": json.dumps(cfg), "filters": "[]"}

def panel(title=None, subtitle=None, bg=CARD, radius=14, show_bg=True):
    vc = {"background": [{"properties": {"show": T if show_bg else F_, "color": color(bg), "transparency": num(0)}}],
          "border": [{"properties": {"show": T if show_bg else F_, "color": color(bg), "radius": num(radius)}}],
          "dropShadow": [{"properties": {"show": F_}}], "visualHeader": [{"properties": {"show": F_}}]}
    vc["title"] = ([{"properties": {"show": T, "text": s(title), "fontColor": color(WHITE), "fontSize": num(12), "bold": T,
                                    "fontFamily": s("Segoe UI Semibold")}}] if title else [{"properties": {"show": F_}}])
    if subtitle:
        vc["subTitle"] = [{"properties": {"show": T, "text": s(subtitle), "fontColor": color(MUTED), "fontSize": num(9)}}]
    return vc

def nav(target):
    return [{"properties": {"show": T, "type": s("PageNavigation"), "navigationSection": s(target)}}]

# ---------------------------------------------------------------- texts
ICONS = ["files", "warning", "question", "typologies"]           # page order
ICON_FILE = {k: "nav_%s.png" % k for k in ICONS}
SECTIONS = {"fr": ["p1exec", "p2queue", "p3why", "p4types"], "en": ["p1exec_en", "p2queue_en", "p3why_en", "p4types_en"]}
TX = {
 "fr": {"pages": ["Vue exécutive", "File d'alertes", "Pourquoi ?", "Typologies"],
        "brand": "Banque de détail · détection de fraude",
        "p1": ("Vue exécutive · le copilote face au moteur de règles actuel",
               "Validation hors échantillon juin–juillet 2026 · le modèle score, l'IA explique, l'analyste décide"),
        "p2": ("File d'alertes · priorisation pour les analystes", "Top 1 % des scores · flux live = août–septembre 2026 (sans label)"),
        "p3": ("Pourquoi ? · explicabilité des alertes", "Facteurs stockés par alerte : ce que lit le copilote pour expliquer, sans rien inventer"),
        "p4": ("Typologies · ce que le modèle attrape, et ce qu'il rate", "Validation juin–juillet 2026 · y compris là où les règles restent utiles"),
        "k": ["Fraudes détectées · modèle", "Fraudes détectées · règles actuelles", "Fausses alertes évitées",
              "Précision des alertes (top 1 %)", "Fraudes trouvées en plus (rappel)"],
        "c_recall": ("Rappel par typologie", "Part des fraudes détectées : modèle vs règles actuelles"),
        "c_cmp": ("Fraudes vs fausses alertes", "Par configuration d'alerte (validation)"),
        "m_frauds": "Fraudes détectées", "m_false": "Fausses alertes",
        "t_cmp": ("Tableau de comparaison", "Validation juin–juillet 2026, hors échantillon"),
        "h_cmp": ["Configuration", "Alertes", "Fraudes", "Fausses alertes", "Rappel", "Précision"],
        "c_prauc": ("PR-AUC par modèle testé", "Champion : LightGBM (score + explication SHAP)"),
        "sl": ["Période", "Niveau de risque", "Canal", "Vu par les règles ?", "Vérité terrain"],
        "k2": ["Alertes", "Risque élevé", "Manquées par les règles", "Montant en jeu (TND)", "Précision (validation)"],
        "c_day": ("Alertes par jour", "Volume quotidien de la file"), "c_chan": ("Alertes par canal", "TPE = magasin · DAB = distributeur"),
        "t_q": ("Alertes à traiter en priorité", "Triées par score du modèle"),
        "h_q": ["Transaction", "Date", "Client", "Montant (TND)", "Canal", "Pays", "Score", "Risque", "Facteurs principaux"],
        "c_cat": "Alertes par catégorie marchande", "sl3": ["Période", "Niveau de risque", "Source du facteur"],
        "c_fac": ("Facteurs de risque les plus fréquents", "Nombre d'alertes où le facteur apparaît"),
        "c_imp": ("Variables les plus utilisées par le modèle", "Part du gain LightGBM"),
        "c_hour": ("Alertes par heure", "La nuit est sur-représentée dans les fraudes"),
        "c_band": ("Score vs vérité terrain", "Plus le score est haut, plus la fraude est confirmée"),
        "t_type": ("Fraudes par typologie", "Détectées par le modèle (top 1 %) vs par les règles actuelles"),
        "h_type": ["Typologie", "Fraudes", "Modèle", "Règles", "Rappel modèle", "Rappel règles"],
        "c_new": ("Fraudes nouvelles vs déjà vues", "Fraudes confirmées dans nos alertes"),
        "c_caught": ("Fraudes attrapées par typologie", "Top 1 % des scores, validation juin–juillet"),
        "c_month": ("Alertes par mois", "Juin–juillet : validation · août–septembre : flux live"),
        "sfx": ""},
 "en": {"pages": ["Executive view", "Alert queue", "Why?", "Fraud types"],
        "brand": "Retail bank · fraud detection",
        "p1": ("Executive view · the copilot vs the current rule engine",
               "Out-of-time validation June–July 2026 · the model scores, AI explains, the analyst decides"),
        "p2": ("Alert queue · prioritised for analysts", "Top 1% of scores · live flow = August–September 2026 (unlabelled)"),
        "p3": ("Why? · alert explainability", "Factors stored per alert: what the copilot reads to explain, never inventing"),
        "p4": ("Fraud types · what the model catches, and what it misses", "Validation June–July 2026 · including where rules still help"),
        "k": ["Frauds caught · model", "Frauds caught · current rules", "False alerts avoided",
              "Alert precision (top 1%)", "More frauds found (recall)"],
        "c_recall": ("Recall by fraud type", "Share of frauds caught: model vs current rules"),
        "c_cmp": ("Frauds vs false alerts", "By alert configuration (validation)"),
        "m_frauds": "Frauds caught", "m_false": "False alerts",
        "t_cmp": ("Comparison table", "Validation June–July 2026, out of time"),
        "h_cmp": ["Configuration", "Alerts", "Frauds", "False alerts", "Recall", "Precision"],
        "c_prauc": ("PR-AUC by model tested", "Champion: LightGBM (score + SHAP explanation)"),
        "sl": ["Period", "Risk level", "Channel", "Seen by rules?", "Ground truth"],
        "k2": ["Alerts", "High risk", "Missed by rules", "Amount at stake (TND)", "Precision (validation)"],
        "c_day": ("Alerts per day", "Daily queue volume"), "c_chan": ("Alerts by channel", "TPE = in-store · DAB = ATM"),
        "t_q": ("Alerts to handle first", "Sorted by model score"),
        "h_q": ["Transaction", "Date", "Customer", "Amount (TND)", "Channel", "Country", "Score", "Risk", "Main factors"],
        "c_cat": "Alerts by merchant category", "sl3": ["Period", "Risk level", "Factor source"],
        "c_fac": ("Most frequent risk factors", "Number of alerts where the factor appears"),
        "c_imp": ("Variables the model relies on most", "LightGBM gain share"),
        "c_hour": ("Alerts by hour", "Night time is over-represented in frauds"),
        "c_band": ("Score vs ground truth", "The higher the score, the more fraud is confirmed"),
        "t_type": ("Frauds by type", "Caught by the model (top 1%) vs by the current rules"),
        "h_type": ["Fraud type", "Frauds", "Model", "Rules", "Model recall", "Rules recall"],
        "c_new": ("New frauds vs already seen", "Confirmed frauds in our alerts"),
        "c_caught": ("Frauds caught by type", "Top 1% of scores, June–July validation"),
        "c_month": ("Alerts per month", "June–July: validation · Aug–Sept: live flow"),
        "sfx": "_en"},
}
VERITE = {"fr": {"Fraude confirmée": RED, "Légitime": SLATE, "Inconnue (flux live)": BLUE},
          "en": {"Confirmed fraud": RED, "Legitimate": SLATE, "Unknown (live flow)": BLUE}}
MOTEUR = {"fr": {"Modèle": BLUE, "Règles actuelles": SLATE}, "en": {"Model": BLUE, "Current rules": SLATE}}
VU = {"fr": {"Manqué par les règles": BLUE, "Aussi alerté par les règles": SLATE},
      "en": {"Missed by rules": BLUE, "Also flagged by rules": SLATE}}
CHANNEL = {"TPE": BLUE, "DAB": BLUE2, "EN_LIGNE": ORANGE, "WEB": GREEN, "MOBILE": "#A78BFA"}


class Page:
    def __init__(self, lang, idx, key):
        self.lang, self.idx, self.t = lang, idx, TX[lang]
        self.name = SECTIONS[lang][idx]
        self.title, self.items, self.z = self.t["pages"][idx], [], 0
        title, subtitle = self.t[key]
        # header bar
        self._text(16, 8, 1248, 76, [[("◆  FRAUD COPILOT", {"fontSize": "12pt", "fontWeight": "bold", "color": WHITE})],
                                     [(self.t["brand"], {"fontSize": "9pt", "color": MUTED})]], bg=BAR, radius=12, raw=True)
        # icon navigation: icon + title under it, both clickable
        for i, icon in enumerate(ICONS):
            x0 = 690 + i * 112
            target = SECTIONS[lang][i]
            active = i == idx
            self._image(x0 + 37, 14, 38, 38, ICON_FILE[icon], target, highlight=active)
            self._button(x0, 54, 112, 24, self.t["pages"][i], target, BLUE2 if active else MUTED, bold=active, size=9)
        # language switch
        other = "en" if lang == "fr" else "fr"
        self._button(1146, 30, 50, 28, "FR", SECTIONS["fr"][idx], WHITE if lang == "fr" else MUTED,
                     fill=BLUE if lang == "fr" else CARD, bold=True, size=10)
        self._button(1202, 30, 50, 28, "EN", SECTIONS["en"][idx], WHITE if lang == "en" else MUTED,
                     fill=BLUE if lang == "en" else CARD, bold=True, size=10)
        self._text(16, 92, 1248, 40, [[(title, {"fontSize": "13pt", "fontWeight": "bold", "color": WHITE}),
                                      ("     " + subtitle, {"fontSize": "9pt", "color": MUTED})]], bg=BG, raw=True)

    def _z(self):
        self.z += 1000
        return self.z

    def _add(self, x, y, w, h, single, raw=False):
        self.items.append(_container(x, y if raw else y + DY, w, h, self._z(), single))

    def _text(self, x, y, w, h, paragraphs, align=None, bg=CARD, radius=14, raw=False):
        paras = []
        for runs in paragraphs:
            p = {"textRuns": [{"value": v, "textStyle": st} for v, st in runs]}
            if align:
                p["horizontalTextAlignment"] = align
            paras.append(p)
        self._add(x, y, w, h, {"visualType": "textbox", "drillFilterOtherVisuals": True,
                               "objects": {"general": [{"properties": {"paragraphs": paras}}]},
                               "vcObjects": panel(bg=bg, radius=radius)}, raw)

    def _image(self, x, y, w, h, item, target, highlight=False):
        vc = panel(bg=BLUE if highlight else BAR, radius=10, show_bg=highlight)
        if highlight:
            vc["background"] = [{"properties": {"show": T, "color": color(BLUE), "transparency": num(75)}}]
        vc["visualLink"] = nav(target)
        single = {"visualType": "image", "drillFilterOtherVisuals": True,
                  "objects": {"general": [{"properties": {"imageUrl": {"expr": {"ResourcePackageItem": {
                      "PackageName": "RegisteredResources", "PackageType": 1, "ItemName": item}}}}}],
                      "imageScaling": [{"properties": {"imageScalingType": s("Fit")}}]},
                  "vcObjects": vc}
        self._add(x, y, w, h, single, raw=True)

    def _button(self, x, y, w, h, text, target, fg, fill=None, bold=False, size=9):
        d = {"id": "default"}
        objects = {"icon": [{"properties": {"show": F_}, "selector": d}],
                   "text": [{"properties": {"show": T, "text": s(text), "fontColor": color(fg), "fontSize": num(size),
                                            "bold": T if bold else F_, "horizontalAlignment": s("center")}, "selector": d}],
                   "fill": [{"properties": {"show": T if fill else F_, "fillColor": color(fill or BAR), "transparency": num(0)},
                             "selector": d}],
                   "outline": [{"properties": {"show": F_}, "selector": d}],
                   "shape": [{"properties": {"roundEdge": num(8)}}]}
        vc = {"visualLink": nav(target), "background": [{"properties": {"show": F_}}],
              "border": [{"properties": {"show": F_}}], "visualHeader": [{"properties": {"show": F_}}],
              "title": [{"properties": {"show": F_}}]}
        self._add(x, y, w, h, {"visualType": "actionButton", "drillFilterOtherVisuals": True, "objects": objects,
                               "vcObjects": vc}, raw=True)

    def visual(self, vtype, x, y, w, h, roles, title=None, subtitle=None, objects=None, order_by=None, desc=True, names=None):
        froms, select, proj = {}, [], {}
        for role, fields in roles.items():
            proj[role] = []
            for kind, tb, f in fields:
                a = froms.setdefault(tb, "t%d" % len(froms))
                name = "%s.%s" % (tb, f)
                if all(x_["Name"] != name for x_ in select):
                    select.append({kind: {"Expression": {"SourceRef": {"Source": a}}, "Property": f},
                                   "Name": name, "NativeReferenceName": f})
                proj[role].append({"queryRef": name})
        query = {"Version": 2, "From": [{"Name": a, "Entity": tb, "Type": 0} for tb, a in froms.items()], "Select": select}
        if order_by:
            kind, tb, f = order_by
            query["OrderBy"] = [{"Direction": 2 if desc else 1,
                                 "Expression": {kind: {"Expression": {"SourceRef": {"Source": froms[tb]}}, "Property": f}}}]
        single = {"visualType": vtype, "projections": proj, "prototypeQuery": query, "drillFilterOtherVisuals": True,
                  "objects": objects or {}, "vcObjects": panel(title, subtitle)}
        if names:
            single["columnProperties"] = {"%s.%s" % k: {"displayName": dn} for k, dn in names.items()}
        self._add(x, y, w, h, single)

    def card(self, x, y, w, h, measure, label, value_color=WHITE):
        objects = {"labels": [{"properties": {"color": color(value_color), "fontSize": num(26), "labelDisplayUnits": num(1),
                                              "fontFamily": s("Segoe UI Semibold")}}],
                   "categoryLabels": [{"properties": {"show": T, "color": color(MUTED), "fontSize": num(10)}}]}
        self.visual("card", x, y, w, h, {"Values": [measure]}, objects=objects, names={(measure[1], measure[2]): label})

    def slicer(self, x, y, w, h, column, title):
        objects = {"data": [{"properties": {"mode": s("Dropdown")}}],
                   "header": [{"properties": {"show": T, "text": s(title), "fontColor": color(MUTED), "fontSize": num(9)}}],
                   "items": [{"properties": {"fontColor": color(LIGHT), "background": color(CARD), "fontSize": num(10)}}],
                   "general": [{"properties": {"outlineColor": color(LINE)}}]}
        self.visual("slicer", x, y, w, h, {"Values": [column]}, objects=objects)

    def table(self, x, y, w, h, columns, title, subtitle=None, names=None, order_by=None):
        objects = {"grid": [{"properties": {"gridVertical": F_, "gridHorizontal": T, "gridHorizontalColor": color(LINE),
                                            "outlineColor": color(LINE), "rowPadding": num(4), "textSize": num(10)}}],
                   "columnHeaders": [{"properties": {"fontColor": color(MUTED), "backColor": color(CARD), "outlineStyle": num(0),
                                                     "fontSize": num(9)}}],
                   "values": [{"properties": {"fontColorPrimary": color(LIGHT), "backColorPrimary": color(CARD),
                                              "fontColorSecondary": color(LIGHT), "backColorSecondary": color(CARD),
                                              "outlineStyle": num(0)}}],
                   "total": [{"properties": {"totals": F_}}]}
        self.visual("tableEx", x, y, w, h, {"Values": columns}, title, subtitle, objects, order_by=order_by, names=names)

    def section(self, ordinal):
        cfg = {"objects": {"background": [{"properties": {"color": color(BG), "transparency": num(0)}}],
                           "outspace": [{"properties": {"color": color(BG), "transparency": num(0)}}]}}
        if self.lang == "en":
            cfg["visibility"] = 1          # hidden from the page tabs; reached through the FR | EN switch
        return {"name": self.name, "displayName": self.title + (" (EN)" if self.lang == "en" else ""), "filters": "[]",
                "ordinal": ordinal, "visualContainers": self.items, "config": json.dumps(cfg),
                "displayOption": 1, "width": 1280, "height": PAGE_H}


A, K, P, R, L, I, X, W = ("pbi_alerts", "pbi_kpi", "pbi_compare", "pbi_recall_type", "pbi_leaderboard",
                          "pbi_importance", "pbi_factors", "pbi_recall_wide")


def build(lang):
    t, e = TX[lang], TX[lang]["sfx"]
    col = lambda tb, c, translated=True: C(tb, c + e if translated else c)
    # ---- page 1
    p1 = Page(lang, 0, "p1")
    kms = ["Fraudes détectées · modèle", "Fraudes détectées · règles", "Fausses alertes évitées", "Précision top 1 %", "Gain de rappel (×)"]
    for i, (mm, lab, cl) in enumerate(zip(kms, t["k"], [WHITE, MUTED, WHITE, WHITE, BLUE2])):
        p1.card(16 + i * 252, 112, 240, 96, M(K, mm), lab, cl)
    p1.visual("clusteredBarChart", 16, 220, 616, 240, {"Category": [col(R, "typologie")], "Series": [col(R, "moteur")], "Y": [M(R, "Rappel")]},
              *t["c_recall"], chart_style(legend=True, extra_points=by_value(R, "moteur" + e, MOTEUR[lang])),
              names={(R, "Rappel"): t["h_cmp"][4]})
    p1.visual("clusteredColumnChart", 644, 220, 620, 240,
              {"Category": [col(P, "configuration")], "Y": [M(P, "Fraudes détectées"), M(P, "Fausses alertes")]},
              *t["c_cmp"], chart_style(legend=True, extra_points=by_measure(P, {"Fraudes détectées": BLUE, "Fausses alertes": ORANGE})),
              names={(P, "Fraudes détectées"): t["m_frauds"], (P, "Fausses alertes"): t["m_false"]})
    cols = ["configuration", "alerts", "frauds", "false_alerts", "recall", "precision"]
    p1.table(16, 472, 760, 236, [col(P, "configuration")] + [C(P, c) for c in cols[1:]], *t["t_cmp"],
             names={(P, c + (e if c == "configuration" else "")): h for c, h in zip(cols, t["h_cmp"])})
    p1.visual("clusteredBarChart", 788, 472, 476, 236, {"Category": [C(L, "model")], "Y": [M(L, "PR-AUC")]},
              *t["c_prauc"], chart_style(), order_by=M(L, "PR-AUC"))
    # ---- page 2
    p2 = Page(lang, 1, "p2")
    for i, (c_, lab) in enumerate(zip(["periode", "risk", "channel", "vu_par_regles", "verite"], t["sl"])):
        column = C(A, "channel") if c_ == "channel" else (col(A, "risk") if c_ == "risk" else col(A, c_))
        if c_ == "risk" and lang == "fr":
            column = C(A, "risk_level")
        p2.slicer(16 + i * 252, 110, 240, 64, column, lab)
    for i, (mm, lab, cl) in enumerate(zip(["Alertes", "Alertes risque élevé", "Manquées par les règles", "Montant en jeu (TND)",
                                           "Précision des alertes (validation)"], t["k2"], [WHITE, RED, BLUE2, WHITE, GREEN])):
        p2.card(16 + i * 252, 182, 240, 88, M(A, mm), lab, cl)
    p2.visual("areaChart", 16, 282, 760, 202, {"Category": [C(A, "alert_date")], "Y": [M(A, "Alertes")]}, *t["c_day"],
              chart_style(labels=False), names={(A, "Alertes"): t["k2"][0]})
    p2.visual("donutChart", 788, 282, 476, 202, {"Category": [C(A, "channel")], "Y": [M(A, "Alertes")]}, *t["c_chan"],
              {"legend": [{"properties": {"show": T, "position": s("Right"), "labelColor": color(LIGHT), "showTitle": F_}}],
               "labels": [{"properties": {"show": T, "color": color(LIGHT), "labelDisplayUnits": num(1),
                                          "labelStyle": s("Percent of total")}}],
               "dataPoint": by_value(A, "channel", CHANNEL)})
    qcols = ["transaction_id", "alert_date", "customer_id", "amount_tnd", "channel", "country", "score_model",
             "risk_level" if lang == "fr" else "risk_en", "top_factors"]
    p2.table(16, 496, 760, 212, [C(A, c) for c in qcols], *t["t_q"],
             names={(A, c): h for c, h in zip(qcols, t["h_q"])}, order_by=C(A, "score_model"))
    p2.visual("clusteredBarChart", 788, 496, 476, 212, {"Category": [C(A, "merchant_category")], "Y": [M(A, "Alertes")]},
              t["c_cat"], None, chart_style(), order_by=M(A, "Alertes"))
    # ---- page 3
    p3 = Page(lang, 2, "p3")
    p3.slicer(16, 110, 300, 64, col(A, "periode"), t["sl3"][0])
    p3.slicer(328, 110, 300, 64, C(A, "risk_level") if lang == "fr" else col(A, "risk"), t["sl3"][1])
    p3.slicer(640, 110, 300, 64, col(X, "factor_kind"), t["sl3"][2])
    p3.visual("clusteredBarChart", 16, 182, 616, 278, {"Category": [C(X, "factor")], "Y": [M(X, "Occurrences")]}, *t["c_fac"],
              chart_style(), order_by=M(X, "Occurrences"))
    p3.visual("clusteredBarChart", 644, 182, 620, 278, {"Category": [col(I, "variable")], "Y": [M(I, "Importance")]}, *t["c_imp"],
              chart_style(fill=BLUE2), order_by=M(I, "Importance"))
    p3.visual("clusteredColumnChart", 16, 472, 616, 236, {"Category": [C(A, "alert_hour")], "Y": [M(A, "Alertes")]}, *t["c_hour"],
              chart_style(labels=False), names={(A, "Alertes"): t["k2"][0]})
    p3.visual("columnChart", 644, 472, 620, 236,
              {"Category": [C(A, "bande_score")], "Series": [col(A, "verite")], "Y": [M(A, "Alertes")]}, *t["c_band"],
              chart_style(legend=True, extra_points=by_value(A, "verite" + e, VERITE[lang])), names={(A, "Alertes"): t["k2"][0]})
    # ---- page 4
    p4 = Page(lang, 3, "p4")
    wcols = ["typologie", "frauds", "caught_top1pct", "caught_legacy", "recall_model", "recall_legacy"]
    p4.table(16, 110, 760, 252, [col(W, "typologie")] + [C(W, c) for c in wcols[1:]], *t["t_type"],
             names={(W, c + (e if c == "typologie" else "")): h for c, h in zip(wcols, t["h_type"])})
    p4.visual("donutChart", 788, 110, 476, 252, {"Category": [col(A, "vu_par_regles")], "Y": [M(A, "Fraudes confirmées")]}, *t["c_new"],
              {"legend": [{"properties": {"show": T, "position": s("Bottom"), "labelColor": color(LIGHT), "showTitle": F_}}],
               "labels": [{"properties": {"show": T, "color": color(LIGHT), "labelDisplayUnits": num(1)}}],
               "dataPoint": by_value(A, "vu_par_regles" + e, VU[lang])})
    p4.visual("clusteredBarChart", 16, 374, 616, 334, {"Category": [col(W, "typologie")], "Y": [M(W, "Fraudes attrapées (modèle)")]},
              *t["c_caught"], chart_style(), order_by=M(W, "Fraudes attrapées (modèle)"))
    p4.visual("columnChart", 644, 374, 620, 334,
              {"Category": [C(A, "alert_month")], "Series": [col(A, "verite")], "Y": [M(A, "Alertes")]}, *t["c_month"],
              chart_style(legend=True, extra_points=by_value(A, "verite" + e, VERITE[lang])), names={(A, "Alertes"): t["k2"][0]})
    return [p1, p2, p3, p4]


PAGES = build("fr") + build("en")
REPORT_JSON = {
    "config": json.dumps({"version": "5.43", "themeCollection": {"baseTheme": {"name": "CY24SU06", "version": "5.55", "type": 2}},
                          "activeSectionIndex": 0, "defaultDrillFilterOtherVisuals": True,
                          "settings": {"useStylableVisualContainerHeader": True}}),
    "layoutOptimization": 0,
    "resourcePackages": [{"resourcePackage": {"name": "RegisteredResources", "type": 1, "disabled": False,
                                              "items": [{"type": 100, "path": f, "name": f} for f in ICON_FILE.values()]}}],
    "sections": [p.section(i) for i, p in enumerate(PAGES)],
}
print("pages:", [p.name for p in PAGES], "| visuals:", sum(len(p.items) for p in PAGES))

In [ ]:
# navigation icons (96 px PNG, embedded so the notebook is self-contained)
ICON_B64 = {
    "files": "iVBORw0KGgoAAAANSUhEUgAAAGAAAABgCAYAAADimHc4AAAwq0lEQVR42u2dd5RlRbX/P1Xn3NQ5557p6ckwMAzMDAxByVGQbHg8UXkiPCWpPBUEH0EUkCdBEQk/BwExEEQERQcBCSI5DA6Tp3O+3X3zveecqt8f59zb3TMdbg/jW+u31u+sdddM3z59qs7eVbu+tb977xL8/2umS+z0AdCA8v4FMAFn3M///9pNIUvA8ARqej9PdwUnec6sG85ehvez9j5iD72cHveC2hsp/5ttzTSalfeZ6goAdcAcYD6wEFgkhJiHoFZrPYjmceBmIDmu7VkpQM7QiT092v63pqoc16YzQ5+qgWagBVgMLBRCzEfQLISoExBECLTWKGdSUb0MnADExw2svKeeBo4HzkCIesAQe1AKGmy03gE8DLwCHAWcJaAJMNizjTkaOoBfAS/s9NsyoMET8kJgIYKFAjFXCFEnBKUIAVqjlEbrCTKcMFMKikOyrKqE3rZ+lNIWggCaG4FveTPcmc0MuA24WEqJUv+aiSClgVKOBv4KHGVIgaP+NRNh3LO/CzwN/KcQYj8ETUKIauEJWXuC3smE5RZXf9AvymtKRXVDhaibWyMaW2tpmF9HXXM1FbVlVNaX8/YL67n23FtVJmWh0e1oFgOZfGe6AL4C/Di7ikvDFFLKPWSaNQKB7dho903NcS/qGFIIQ4o91BIIAY6jcZTW49YZhBToiYLOIRYppSipLBLVDRWivqVW1M2tprG1jvqWGqobKymrKiFUHMLnN5FSorXGth0cy8bO2BSVFfLVo67SH/xjk5CGTClHLQba8zXrJnC5EEIJIcWnL77ROOjY08CB2mKB2ANS0Sh8Poff/PxufnXvLdowpFJKG9ectdz8zBELkUohKoNMbEzvBoYApTXaZ/CL33/A9fe/iRBCK62FVtoOFAZlVWOVaJhTSUNrndHYWkt9Sw01jVWU15RSVFqAP+QnawUc28GxHJTW7r+O+52VsZFSIg2BchRKaepbasQH/9ikhSDoLdbt+Y4pIaXhKOXIpvnLuO0P72PbYApoLHOfoBzQmryVkb1XGp5QFAQCMDpscfohjQwPDTC/toQtd54BhrdG1oa8xvTsJl72XinGfjYlpG3mnP4LOvpjAHrlSWvEp7/7RUpKgsyrMzF9bufsjI2VsdFKod3ZgGEaGKZESoHjKNKJDJHhGPHRBEVlhdQ0VZJJZcik3b8rqyrh/1z3G37xg0dswzRMx3Y+D9zvDW47nxkgXXsXIpNWZNIavylIJV3pFZaYGMaYIqYUjid4wwDLgnjMxpASjca2BJaVIRBwYXNBwMSxHETGQRsCkbJRaMxCvytAR+fZmABbYcczSCHQAkRGYDuKUMDMDbKjzj2BqqYa4uFRoqNJDDSOUhSWhAiE/GRSGeKRBOG+UQa6huhtG6BrWy892/vobR8k3DdCOpEmVBRk5VH78oWrPkVlbRmJWBKlNA2ttd7g03iLe96XOTZyNUJIhNQIIZBSU1hs8uK6dTz31CMMDfR4C7Se0hxIadDQ3MIxn/wsB6xZTSyqELijyZ1BAiEEGo0U7nfaE5xZFOSFV9v4zbNb6ByIo7xROZXhkVIyp6aITx29gENXz0HFM0iB23clJiCY4d4wjmXjKNBCIqWiqKyYt55/n3W/fon2Td0M9oQZHYySSWemFFYqmeYvv3qRDW9s4ZY/XEVhaQG2ZVM3p9qVobvOLJqNHTUnNyOKYFBy05Vf49f3/WjWpv83P7+dr15xE1+45BtERx0MwwAk6VQCrTXJzBhC00ojC3xcdcfLXH//G7Nu68ePvc81563m6gsOwolnMIxd7+lv68XwGWggY2nKq0M8dMsT3HfNwzMgN0lFRQWNjY3U1NTw5ptvMDI6SueWHh764eNc+qP/IBFLUlVfTqgoKJOxFEKIVk/5arcUoGyH4mqTxx5cy6/v+xHSMFCOg8/nR3qIZSqLYDsOjm0jpeDHN1zOwr3259CjjyQyalFSFuTjx53OM4+t5aw1LYigj0w0jb8ixKN/3Mj197+Rg48+00AacrrJhnIUlu1gSMF373uNFQsqOfmYhajYriO4f0cvWrkbKH9hAW8//x73XfMwUgqU0hQXF1PfUM+c5jnMnz+fRYsWsXDhQubNm0dDQwPl5eUIIXj77bc57LDDSCQTvPnc+8RG4khDUlJRTGVduejc0gOCJjTFQDQfKLqLAqQhSSUUj/7iDqSUCCH4/ve/zxlnnIEQrgmZ7FJK4TgOP/3pT7njjjsQQvDoL37MYcceiRACJ6O59Ac/01ecusRexT98OuNgGgIcxe2PvO/acOCqq67inHPO8cygnLItpRRr167lpptuRArB7Y+8z8lHLXT7pycOvoH2PuyMBUKgpcHTDzznmkINl1xyCZdffjnV1dX4/f5pR+uKFSvY/4D9efFvLzIyEGFkMEJVfTn+YICapirRuaUHIUSVRtfvlgK0BsOUjI5EGOzrQinF0iVL+Na3vpW3Sbjhhht46KFfMjQ0SHf7NpJxMAwTpRxsHObUVWunw0b4XGWno2naB+IorWmZM5drr70277auv/56HnzwATo7u2jrj2HH0ph+Y5fJH+4ZJBlNYPp8xBMWfe2DaK0pLCzkyiuvpLq6etp20uk0vb29tLe3Y2UsABLRJIM9YWqaqzB9Bg3zanjreZQQGN5Oe1M+eG7KNSC7iBUXF8/KJhcWFlJYWMDQUPY5Y8otKDQIBYMYQRNle6ZE69w9s23LMAyKCotyz1d68sEWGRglGo5Q3lCNlgYFJQU5wSYSCXdn5jgMDw/T0dHB9u3b2bJlC5s2bWLr1q10dHTQ19dHLBbLWQnlKHrbBtjn4CVoDQ3z6jy7LySwAPjzbivAXRzdYbRp40ZOPfXUnf0ik5tmIbAsi4GBgfGwDKUUBQUGf33qaXHjAz80v/bxEg5bXoTO2BPua2tr45RTTpnSzE1mijo7Oyc8YzK3hJXOMNwzRFVzLUIa1Mx1YaOjHC666CKEEGzZsoW+vj7C4fCM75rtX/f2PgxDYluKhnm14+W2aNYwdNdZ4HZiZGSEJ554Yvc3wxoMQxKPZ/jRdy6gt7tDbt9exbs/bMpZyGxbkWiUJ598cjfb0VPtNEE5DHT0s+SQfXAU1LbU51DOTO0VFBRQXV1NU1MTixcvxjRN7r33XlcB2/rQChzbpqa5EiGF8HxLC/KFoubMTjTpwcj8L0cplOOMEwJk0hm01hiGxHEctOUgTLnTaJXIWbalHAdnOgeiGENCUkpsy6Z6Tm1uBkkpkVJSVVVFY2MjLS0tORTU2trKnDlzqKurIxQKuXuK4WEeePABkokkve0DWBkL5SgqassoLi8SkaEoQoh5WmuZj0d0SgVkBbF0r714/LHHvI2amHYECiFIJBIcd9xx9PX1Tbjf/b+7vRc7uXGyaKelpYWnnn465/Saqr3s72zb5sQTTmBHW9uUiGn8XkB7Pp6KhioCfpN0xmb/A/bnoYceora2lrKyshkVXl5ezpw5c9j44UaGusPEI0kCBX6Kywqpqiv3FECD1lQAgzMhoRlnQDAYZOHC/HfX6XQan883jfNm6t4EAgEWL148qxkwE3TMmqbBjn6stIVyNEVVZZRUFjPQM0wqlcqrzdHRUbq6umhra8sRMsMDEYYHR2loqcUf8FE7t1ps+6BdI0QZ6MY9ogAh5awEEggEdp8q2w3364x/4736SO8QiUicQGEBweJCKurKGegZpr29nWg0SnFxMfF4nP7+ftrb29m6dSubNm1iy5YtbNu2je7ubsLhMJZl5drNpDMMdA7RvKAeaUga3YVYCSEMYB7w7kxIaEYUtHXzZk4//fS8TZBt24TD4WkXxqlGaUdHB6eddpq3SZrZBCml6O7unrat7PfRcITI4Ci1JYUYfh/VzdVsfHsbiWSCT3/600SjUdra2hgYGCCZTM5MMBkSx3bo3dGPYRoopWhorcs54cc55cRHQkHDw8M8/vjj/1LidjwK+t3vfrdHUVCWkFGOItw9SMOiZrQQ1LbU5n7/9NNPTzvDKioqqK+vZ+7cuSxZsoRYLMbd99wNQNf2PqQQWJZDfUu1t7jrvKFoXihIztIMZV0Fs6cS5axNnlZqehQECCHROAy092GYBo6jqW2pyynONE1CoRC1tbU0Nzfn/EELFiygtbWVxsZGKisrczPyww8/5Gd3/wyAnu19KKVxbIfqxipMnylsy0YI5me3QR8NBS1dyqOPPjorFHTCCSfsgoJmUnIWBT35hz/MCgV94qSTpkdBejwU7XE3ixnH3ZR5g+XEk07kpz/9KVVVVQSDwRn729raSlVVFYMDg/R1DJJOuRC7vLqEsqoSMdgTBsRc0D7Amm4hnhkFhUKzQibTo6CZF/ClS5fuURQ0fiXub+tz9w2OQ1ldJQVFAeKxNLFYjKampmmfkMlk6O/vp7Ozk82bN+cGx2B3mNhInMLSAgpKCqhurBCDPWGEEPVa6xqg6yMp4P9hFJQd+xGttQEUDnUNkEmm0UgKK0oprSwhHhtg27ZtuT8aHBykq6uLHTt2sHnzZjZt2sS2bdtob2+nt7eXaDQ6YWmNhKOE+0YoqSjC7/dRN7dGbHhjixaCEG6cUdd0C3EeKGgLZ511Vt4C+SgoqLOzkzPPPHNWvqDunu6Z2hpEYwGLR/rCKj4SkwXlJQQKQ1Q2VNDdNkBffx/HHHMM/f39dHV1MTw8PPMapskR+H0dg8zfZy4IsvSkgxAmbiTdq7ungBwKCvPII4/8r6Cg0UiERx99dE+hoOwMiGutu4DFyUhCj/QPU1xVhjBNaubUwt83YlkW69atm/LZoVAo5w9qaWlh2bJlbN68mbVr17oL8Y6+HBRtnFc3vj+L9gAKEgghZy2M3UFBUojdQkFq+plmAZullMcrpRjqHKBl3wUoLXJISAiB3++nvLw85w9asGABixYtYv78+Tl/UEFBQe6hL774Ij//+c9dKLqtz419tBV1cyfwwwtncsrNiIKWLNmLRx75LUqpGVGQlJJ4PM4nPvGJ3UJB8+bN44nf/z7vjZht25xy8sm0tbdPB5UFsCkbV9Pf1utuohxFjacAx3Y47/zzuPHGGyktLZ22Xdu2EUJQVVWF6TOxLZve7f3YtkLZDlUNFQQLAiKVSOfFD884AwoKCmaFTBzH+UgoaO+9997Ti74JbM3qun9HLwKwLIfKpmoMKXGUIhKJUFZWhuN5cbMmLUvDZj+maSKEoKmpiarKKnp7e+nvGiIVTyFNSUllMRW1ZbJ7ex8ImtEUAImpkJDMd1PlOE7u/5N9sr+Px+N5L75TjbCZ2sv+Lp1O59OWD2gDnQHEQHufdmwHx1GU1lRQVOq6mTdt2pibxYZhYJompmliGEaOG88qRGuXyG9sbHTXyb5RIsMxhBCECoPUNFVm760B6qdzSch8TcRsPnykIN493pYf6NFK9wGEuwdJxZOgNQVlxZTVuC7oru5uotFoTsAzDUqtNa2trQDEIgkGu8MYhsT0GdS31ArXKYcJzP3ICvh//DKAUdyQdUYHRlQ07AraDAWpanRH69DgEF1dXbsgqiygcBwH27ZzHyEE8+fPz3How/2jSNNdNz0oqjyZL/hICshi3Xw/M7ksZmor+8L5tpXHLDDcuF29DSHIJNN6pHcIwzQQ0qDW44dt26arqwvbtslkMrm1IBseM94sBQIBwuEw69aty71rSUUR2lFopcfxwzNDUXPy3aWB9KBnLBablVkxDINEIpnbRQs5DQjTLvTMxtbG4rFZ0Z8+ny8XqSCE+6xprk1SCpSjGejoZ9GBe6M01Myty93w0ksvcfTRR2OaY2KJRCL09PTkdsabN29my5YtvPvuu7kZM2dRA4v3n086mSFYEKB2TnXOXT4TFDUnCh5sW1FaXkRNfTMj4X42b97E1VdfnXdg1l133cXwcNhFCi0LKSiAeGySQFvhuogDJQFa6opo64vS0d7BN77xDc4555wJC99UNnjt2rX09LgOttb6YsziACoxZWzn5my6wMCOXgxDkk5lWLxqUW7g3HTzTYyMjFBUVMTGjRvZsWMHnZ0dDA4OYduTBzoHQgEuvuU8AiE/8UgS23aorC2jqLRARkfiWX54yhQpc7KNTTBocvYXL+Xay/4NKSXXXXcd1113vRuaOFnQssbTuJNTpNZw9hcuwZ3JYsqICQzJZWcv57l3upFCcMstt3DLLbcgp5k5wg2oy23etNZcdvbycc2ISTgxtmVzMvraegFIxlIsWrmYE875OH988AWSiSS33XZbHjPPpKSymEX7t/LZr53K0pULiEcSGKZL0pRUFFFZXy5cBdCoNWXA8GRQ1JxsAxaJKE7+1Gdp2/pP7v/x93LvkUvn0VMIM2uGTB+XXn0rB37sUKIRB5/PmMJcCVQ0zclHL+SGtmGu+Nmr40b5zDRj9rr5Kwdz3OGtOLEMhrFLmlV2yLSjSQKhwY5+baUzwjANotE0F/3wC/gCfp556AXSqYynWElhaQHlNaXUNFXSMK+W+nm1NMyroaapisq6csprS9FKE48k3FhWb3aGigqoba4WOzZ0ghDlHj+cnwKyjScTikuuup41h5/I8396jMG+bg8d6CljP6SU1DfP4+iTP8WyFfsQjagZ1w8hBSqR4dvnH8jRK5v47V+30jUQcyPmpgtPF4Lm2iLOOnIBB+zXgIqlkVJMpS4fMKC17gFah3uGdCKSEL6CIBnLQiu45Efnccp/HMPW9e1IKahpdoVcWlFMsCiA6TO9MBiFbTnYlk0i6malZoWfXXhN08guxM44fnj9ZKbAnEYyRCOKlYcczEGHH4xWM0cZuXH7kE5DdFQhDZnXpkwKdyasWl7PqpVN7nTKqzEBGQcVTU8due0+yQAyGt0GtEaHIjoyOEptayFOxsZWithInKaFDbQumwtobEvh2Da27ZCIJnM5Zrl1UDDF4HL74UHR7ItMCUXNmWBhPOZ4Qsw/b0gIOWFU5AdBBSphoXQGMVNb2Yg6L9FjipG/K9zWbBVCHOHYtg53D9KwuBkrBUoLfIYkk8yQTqRzC40QYzBUaRfWTwdEcuuTrahvqc2LHzbzSS+dyucDGimN3cb9kylBTiJ8x8vZNeTOLz/rdjcJKdCOZqCjD8MwcBTYCvyGaw4nU74QgsLSAtCQSVtkUhaGKackiBzLpqapEsM0hGM7IFjAFPzwbu+ES8oMyipMTFOg/8V570ZxALMshDANPmJjOSjav6M3p0xbiRndDj/7zkN851M3M9gdpqy6ZOqMeQG27VBeW0ppZXE2MWuuN9jVzqNGztZZ5jg2pqlZe8dt3HD5ZURGR5DSye0c9yRJYzkKfAZ3P/wOF1//LP3hBI4hsZ1Zcw3ZP9juaVD2t/Xm4ldtZ3IF2JZNRU0ZHZu7+e0df+DVZ97ix99cy4Y3NqMcRVFp4S68hxACx3YoKimkqqFCeN81AFWTTVtzNqEmpimprjP52zOvcvt1l7ru6sJirrjpWoaHIJVUs7b9k5MsGmFIfFUh1r/ZyZd/+ELORN167XEwmkSl7Hxs/87AtUtrPQqUDXX263QyLaQhsSbBvEopisuLeP3Zd7nvml9h+kykIXntz+/w2p/foWVpE5/71pl87NQDXRgqJyKhQMhP/dxqsentbVoICj1+uHe3ZoBSmlCBZHS4n2sv+zpXffVsfD4//kCQ395/O18+4yzef/MfFJV+9FIHWmlE0GQkkeHbP3iek//rafymJOQ3+T9PbeDU8x/hxbe6kEWBncsM5HOF0XS5LuRhHR+JYZgGtnJTlMcirhVFJYW88NirfOMT1/HBPzZhWzaZ1Ngue8eGTq4991ZeefpNCksKJpgj10clxpI2XFPXulszQClFICjoamvjq585mq62LRN+n0mneO6Pj/C3P/+O637ya4477XRiEWfKxXvatrRG+A16B+Mcc9ETrN8enmhFMvDEi9v5/Yvbuf87R/Hvpy3DiaYxZp4JWRintNbbgb0TozE92j9MSVUZdspCaRfVaq3x+X0MdA/xo8vuzSm5obWWg47bn1BRkLeee5+Nb21DSMGDNz3G6qOXI8b3wdudZ6HodPywzGf2+nyC//nupXS1bUFKg0AwxKpDj+KwYz5BVW2Th4psvv/NL9HfPYTPJ3eLlNEKRMDkm3e8wvrtYQxDEPAZHHFAE6ccNo+W2qLcGLrg5hfobh/B8BvkORGyEtoipUQrrYe6BjB9LhJylAs7ldIECwO89/KHxFxfDgcdvz/3vHIT//mDz3Hedz/NLX+4isb5dTi2QyqRxradnULx3YW4bm7NZPxw/jBUa43PZ9DfM8r7b76EEJKq2gZue3Ad85csQgiIx0b5+ufP4O1XnyM6Guad117h+NNPZnTYmZVnUwOGT5IaTvLc211IIagsDrLu1lPYZ68a0BDP2Pzbt/7Ik6+0kUjbPPdmJ/92xj6odHI2a884frjPzffSYGu33syu/dIUFocIFQZxHIWVtggVBfmvOy9g3W9e4vDT1+Dz+0gn0rlZ4EJRh+qGCgJBv0inMggh5nuD0pn1IiykRBqmt8kSlFXWEAzB6DCUlZdy1hcu5s1XnsXnD9A4Zy5WZveCrPDMgOFVKxISqsuCEPTBSJLCQj+XnL2cJ17egRSC+c1lYKl828pKd8sYP9wzhngc12EhhCCdTLPXqgVuGYO0xbO/fZntGzpYfcwK5u8zl6b5dSxa0cqaow8gPDxCMubywdkWBALbciitLqG8plT0tg+AYA6aIJAa7xMyZ066c6iuLWbpvqt5+dnfM9Dbybkn7MunzruYjx17GoHgfE44/RQKi16gsKiEJfvuSyqpZ01NZsM6/OUhDl1Wx4M9mxkcTbHfF3/DxWcv58zD57OopZwjDm/l5TtPw2dIVi1vQCetfNaA8QpoA+0AxkB7n7YtWwghsNTYZjCTsqifV8t5V3+aO7/9C9eVur6dbevbXW9owKShpZaPnXogp194AsGCIFbaGlsHvETywuIQ1U2Vord9ACFEjUbXATvGK0DmIxrb1lx05U2UVdaglKK/p4M7rr+czx2/nEvOOZE/Pvo4hx//MZav3o9EXE9ckGajBAk67XD9hWtorinCcTR94SRX3vUq+537a4696Al+/cQ/OfjgFlatakIlrd0pqdOttR7I8cOxFNKQ2ON8XdKQxEcTnH7hCVxx30Us3G9ezhkHYKVt2jZ28cCNj3HFmTeSiCbdUgjj1j2lNT6/SX1LTZYf9k/GD+dFSaaSmnmLFnPXb1/g4CNPzE35ZCLO35/7I986/3QuPPMUwgPD+P3sXBxpVoFZpG3m1pfw/J2ncdrHW3PmKJm2+cvrHXz66mc49kuP0N8bRQSM2S72Aoih6QQY7R9WseEIhmlgORM32UIK4pEEn/jCUdz795u5/S/X8LXbv8RpXz6evVcvQgiBYRpseGMLD/3wcQqKQhNhsUfNNrbWjeeH589aAVklCAF7r1jCXY88xd2Pv8pnz/86zfMW5kzVS88+yfVf/xI+v9jtsJTsiyMErQureOzHp/LaPWfx7c8dwNKW8hzC+MvrHfz7d/+CntoDOh0/jMsPQyqe0sO94TEkpMc8QVprAgUB/vTA8zy1dh0L95vHiZ87kq/cdC7/88fv8v1HvkUw5NbPeO0v7xAbjU/wD7kE1Th+eAooKvPbB8DvH36AC878JC+te5lVh67ma9f+kAf+9DZX3HwfpeVVGIbBC888yj/f/YBQwe5tyJTWEDR55M8b+eR/Ps66v2xm/xUN3HDRIby19lM89N/HUFdRgGkI/vx6B2+8040s8M+m9pwY44fdVx/sdFOMXCjq2W+lCBUGef+VDVx77q3cdOFd/OmB593yNBmbguIgTQvqMPwmWrsmKQdF9ZgCbMuh1itl423UduGH84ChkvDgKLdeexGxyCjDQ30cevSr2BkIFhRy0Te+yPZNG3j4nh8ipKRzxyb2Xbk3ibiedc03Kd3aERff9hI9Qwm2dkdY//FWcBRBv8Fnv3wQ2ztH+c69ryGlYEP7CKsPnDNLd/kEpxwDXqii1q5TLohbOkEakmTcdU37/Cb3XfMrRgYj1DRV0b6pi6fv/yuxETcIba/VCymtKCYyHMPIwmHhRlpU1pVTWFIg45FElh+e4BWdAQV5JceCQYpLKkgnk2z+5zv84Jvf5LjTzsY0Azz+4Muse/JhDMPEcWyqahpxbGa9OIpsupDfoKIkyNBoik0dI1xyw18556SlhPwGrz+9gXv/sAHTc8g1VxWCM6swmOyLb/UGodG/oy8Xij+GhCSJaJIDjtiHpSsXsOGNLYwMRrjvml/t8sCqhgq+8J2zSacyE6IysnuBkooiKuvKhasAmrSmBIhkX9mcGQE5lFUEOPPci7nje5chhOCX99zEL++5KUe+Z6+VhxzD8tUrScTV7rkiHI1RGuCys/flP258HiHg9kff5/ZH398lYm3N3rUctroZlcjkC0PHT/12NCkgONDRp620JYQU7l4gN/A0Pr/kqrWX8KNL7+X1de96fLdby0gIWPHxZVx4w79TN7eaZDy1C/RWSlFYXEBNsztrEKLS44dzCpg5qsmQxCKKcy68lHO/eiVSmuOCqMZefM0RJ3L9nb/0vts9GJol6c87Yx+uP/8gAl6kmSEnCv/IAxp55Acn4jPkbPmB7M19WutegOGeQZ2MJjCMiUgoux8orynjul9dzq1/+m/+478/A1pjpS0a59fzg0e/Re2cyYWf44fdUjYeFBUSt5RNbj2S+RkHSSatufg71/PLZ9/h5E99yQtXhyX7rOQnv36G/7n/KUrLqsik+UgMmRQClbS58vwDefeXn+XiM/bBUa6Z2ae1gqduPYVnb/skDVWF6LQ9UzDWVP6vjEa3A0QGR3VkaBTDt6tXVEiBlbbIJDPstXohn7/yLD516Sk0LajnzP88EStjk06kp9l0uihtDIrumj8sZ2OlI6M2i/bei4uuvImq2gYAzrv0Go4++VjSSRvL0rPx0U+jBLAjKRbNK+fGrx5Ca4PLQF157kpOPH4xtuWgM/bubvjG88PYGVsPdw9hmrtC0awShBQkYymGeoc594qzuPO573HMZw4jlUhjmMb0a6gzxg9P5pQzZ9Nz0zQZHXYoLi3j3ifeIDoyzIK99qKn08EwTfbkZZoSJ5oh6Dd49a7T6QsnWbaoCqcngmHIPVFuehw/nM12d/lhn7Fr9E3W2ZdOpJGGIBlPzzjYXCjqlrKRhhQ7QVE1awVkQ/gsS1PbUE9Dcz2ppJ51OZvZrAnadqiuKqS6thidssZg3kercrwTP9zjljf2+GG3sObUFtm2FWiNo7xN6lSKEGBbDhU1ZZRUFImRgQhCiLle1qYDiN0etpmU44aFyH+N8MfPYyfjoLVbCHYPnTEwIVRxAj+spg6jVErhD5gEQgF3IAhBKpEm47qbJ4+QsBVFZYVU1VdkFdCota4E+mc9A7KdNH0GRsAr/2u7Wp6uyuHuXNndrWFKjGxoo6NwLHfx/whrTVYBnVoTAUqGOgd0Jplx+eFJCH+tNNKQFJcV09cxyIbX36FjSzd2xmbNCQfQsrSZTNqatE9aKQKhAurmVost7+3QQlCMW7a/H5BmvhEKWmuKSgyEgPBgktHhQYQQlFfWUFbpx7YgHnP2iDlSSmMU+kEKUsNJBkaS7gkL5SEC5SFwtFuk9aMt+ENo3Q2UDPcO6fhoTISKi7CVmsgPK40/YGJbDvde8zB/vP85wv0jud+v++1L3PfazZh+Azvj5IK5xsvOMETWJ6RwQxVbgbfyMkEuwQwFhZJXnnue3z10F+vfepWRsKuAiqpaVhz0cU7/3IWsOHAl0VEFyN2qvJ71KMiSAK+81sFPH1/PS+/1MjDi5hvUlIU4bHk9F5y2jDWrmtCxTK5o7Gy9Hm6pcL0DWBIfjunIwAjFFaUT+WGl8ftN4pEk/33O/7D+1Y27muKw4u/3/ZN9T22loDwADjiOGneYihs31ZgtZbNTfWlzZuFrpKG5/hsX89gDd+5yT1d7jK72rTz9yP188ZKrueC/riYRd7x3FLOSihYg/QbfuPkFbnn4nV3u2Z602N4T4Rd/2sg3z9mfH3z1EJyMjZy9Etza/potQgqUUnqoa5A5y1pJJ1woagiNFi4Evf6Lt7P+1Y0uUnIU85pbqa9pwOfzUVtdy+bnu+haP8hRF++PUawpKinKRZO7oYpj/PDOoYrmTK4Tf8Dgigs+y7onH86egkFBqJDSklK01oyMDpNKpwDNvT/6LslEgq9f9wOiI86sCvAprTEK/Jx/7V+458kNuTLGBcEQpSVuIt1IZJhkKoUhBTc++BaxeIYfX3kkKpbZ3c3fplwpmzY3aUNpF4qa2qGoopjf/ewZ3vnbBximgWn4OOLQI2mqb87tzG3bRgcUdkzxk68+xPub32XVkftx+Z1fxklbOXqyurESX8AnrLQ1oZTNlApwbIfyCoMHf3Y36558GNPnQzkOK/ddzaLWxQT8ATSaZCrJBxvXs37Te5imj4d+diP7rzmSw084ltiok1fmu+MojLIgD//uA+55cgM+08BxFCuW7c+SBXsR9LslZFLpFBs2f8C7G97BZ0p+8vh6jti/kTNOWoIz6uLz2UNRJiRtZCPljIAkFU/z1NpnXTTjOHxs9eE0N84lkYhPTFtVGr/fYNOmTUTCMZ797Uuc9IUj2WfNEhLRJLZtU1ZdQnl1iejvHAIh5qK1H8jIqUyP6ZMMDyV54M7vIbxyj4esPJT9912JaZhYtoVt2wT8AQ5eeQgr9j4A27YQQvDz26/BtvKnJqUhseIZrr3/Dbf6lO2wesUaVi0/CJ/pw7ItLNvC5/Nx4Io1rFp+IJbtkvHXrn0DJ2G5wtezRkI7vELTcqCtVzuWA0KQccAf9NG1rZf2D7vQWlNZXsXcphaSyUQufWrnqpBzmlpy37/38gZ8ftONIbU9frixMhsrWgvUTumKcCvdCt59/e/0dbejlaKxrpFF85cSj8fQXnRENhEtmUqy79LllJWWo7Vmw7uvsfXDLQSCMufqnQ7xiJDJO//s58O2EZTW1FbVstfCvUkk4yitJrQVT8bZe/Eyqiqq0Vrz/tYh3t84gAiaM9WMmEwB3VrrIYChrkFSiRSGIbFskKZJuG8Ey3JzwyrLKjEMc9qIvqryKjQuYhzoHPIaES4/HPRl+WEtBAHco06m8AVpjWHCtk3v57TcXD/HC0vZdbOhtMJn+misbcwFabVv3eDywzMIRWsNpjEhCq6pvhnhnb4x3jOTPfzBkAZN9c05SX6wYxjMWfHD2Rsj4/hhHRuOYpgy55Qbv69RSs3oeVXjslgmmF7t5jF4oYrO+FBFOZ0TNJUcKzvg9/u9h09tVgL+QE5QqVTCuzU/ocRTVu7Jfn8gdwLTrt1yXQYBr1KW8P52N5xDY/wwkIwm1EhfGNNn4ijIZOxsYJVLXYYHsGyLqYqdG4ZB/2Bf7ruGeTVel1yINtVRJ3Iq4WsFFZVjlQUjscjUafx67J5sA2UV1V6inchrA1BTXjA2LKOjrqAnaUtr7ZbBikZyTdeUhXYnbyDbsc3ZkT7UOeBCTQTJhE1jay0Llrt2fSQywqZtH1JYUOjada3cj1IEAyFGIsNs3PphziOw38f2JpOycgENju3kjjoZ75STUwdkwZLlB3r5UIJt7VtJZ9IYhpFLWshmtft9fiKxCB09HW42SVEJ85csJ5OaueSZ9PK8Vi2uxme6nd3RuZ1kKolpmru0ZZo+4sk4bV07EAICfoMDFlVD2tkdbsBFQmJcKRvp8sOW47pcTv3y8e5u1jR4/d3XWP/he/hMH6FAiFAgRDAQZGh4kHUv/oW05RYPWXPC/ixeMZ9kPOW6sz2vaGV9OaGiUDZnYN7UCpAGybhir+XL2HflYaA1kegof3/zZQzDIBgIYhompmESCoawHZuXXvsbmYzbgcOOPY2mlioyaWdGfC6EQKUs5i2o5NhVzWgNiUScl19/EYGY0JZb0VDz8msvkkwm0BpOOmgOza0VOGl7tnuBnUMVjf4dvblZ5whJbDTO4aev4aizDsFLNeLlN17iyXVP8Orbf+eN915n3Ut/5sl1TzASDaOVprymlAtu+HcyaWsCVLVth9LKYipry7wvaQKKzcnJcXdjJCV89cqbOf+0NW6N/R2bicYi7LVoGWUl5Wg0Q8ODrP/wPYYjI6A1hcVlnP/1a0in8iTLvYxIrTQ3XriGdW90krEVOzq384dnf8+yxftQXlqOBoZHwqzf+B7h4SEQgpDf4IYL1qCtXQuCz44f1hnAP9Dep+3MGD8shCCdSPO1O85HA3/97cu59WAwPLDLA+tbarhq7SXUzakmEZ0YMKwcRUFxiJrmKtG5tQchRLVGN5mTeyJBYRCPOhxw8GquuOk+vnf5F1wydbCPvsE+TJ/PPaVz3MLkD4T43p2/oWXBXKKj7k54WmTiHd4mlUYlLfbeu5YHrj6az1z9ZxzvRZ//+1/xeayTZY8FFpsSHvruMSxeXI0TSbvcwe55RXs1ug9oDvcM6WQsIQyfj4yjcuUUJJJv3/0VDjh8H3539zNsXT9WwBugqr6Cj592IGdfcjIV1WW7CH+MHzZpaKkRb7lQ1A/Mm3QGKCCZgfICg8iIw5mf/zxVNbXcfv032L75n17+1EREsPd+B/K1a29nxUGriYzMwiuqgZSDDJo4kRRnnbSE6vIQl932Eu9sHtxF8AD7L6rmR5cdysdWz8kJ/yOcPZxC63agOTIwoqNDEaqaasikFWlbEDBdCJqMpTjhc0dw+Olr2LGhg+4d/W4p/NoyWpY2Ud1QQTKeIhGbPlR+rJQNBrDYnKpX0QyUBF0GbHTY4ePHn8CKNUfy0ro/8MbLz9LbuQNpGDTNXcDqjx3LmiNOIBBg9sIXApIOFCm31sJomsMPmsPr+5zFU6/sYN3rHWzrjiIEtDaUcMyqZk48eC5GyPdRhD8eitra5YcPsdIZHe4Zora1gXTKYjRtUGvaucTs6HAMaUgW7jePpasW5kxLOpVhNBxz02ynEr43m3YqZbPUnIpMthwYSUFVAWAYREYdTDPASWeewSfOPgPLg96mzz3mMB6DeEzNng8QuN6vqAXlAffYxFgG0xB88thFfPL4xe7vs3ZHaYhnUPHMRxX+BCSU5YcHO/oxDAOBJp6RjBqSsqDC8aLlNLgRczo15nGWYheqNLsmjY/CdWyH6uaq8UedLDKni0wYTYJPQmnQVYJSmtERlcuGx9t2C69ewm4zYlJA3HJPES32I710JRVJ57LhczlkXmb8noi+mJA1s0v+sEYKCCcNFFAWULlANC2FtyXcSchZt7qXceM4YCmBrcCyJTotSBdUUVhaJGLDUYQQC2YMTRyMu4tyWcjLZBfGhPO65R7kfhnNuNax2IcwhHestxjnxN+jQh8fqpjjhwfaer3AYpHr1nDSIGlJSgKKgKmRwt0MOhocR2ArN8HDVsL74IW4uANJ52pfWBSUl1JWVyFiw1EQos7MRy7hJCRsKA242UKGzF/w2hvgeQ1YISCSgZQDRSYEDPfE1NngezGr+3P8MJoYUDTQ0a8zqYwY78mVAtK2oN82MAQY0nPma5ezUGq8z8urX2EIfD4D02e6sUNCYPpM2j7Yxkhv2I3Z0jppTkhHmwIySgFpC/os1wz7Dbcqisj7GCtIxPWEGkBTnlEshRslG067jZme9vISqvtQbXvDL/8rW8pmYZYfLiguwMmFnHt+Ta2xlRuIgBcYYJgGAU/I2QXYthzSiRSj/SOM9A8z2NFH/45eBjv7+eDFd3VsOGojhE9r/UdzfFEOaQicpDVp6Zfs61sOZKxZuN69pGXH8aE9IWZHiG05k7jcxmnGctzT2WeJK01D5Gussvyw4/HDC2PDUR0fjlFYUoiyHaQnWNM0MXyGu0ALgXIc0sk0idE4owMjhLsG6G/vo397DwPtfTrcM8hI37BORhNqJ9QlEPg86Hu5qbVOgggO9baL0cER6lvKvCzHGQ4Rz3NAKg3BEKx/bQPREZfI7wrH6RlKUN9aARkH9lC1FbQGv0n75gH6RpMeH65TXv3omfjhrVLKo5Tt6K1vfsjCVUtIjMZwLIdENEF0cIBwzyCDHf30t/W6I7qjX4/0hXVsJKr1GBkhxy2NbgklY4wf10rFtNbPAl8H2gTwJyHEcVprq3nBPr7lBx+N44g9VoLGbdXitb8+oQd72pSUUiil9OKGUnn8Ac3C8Lybe+qyBTz1egdbeiMZIfBrzV3AhdMcMZ6NUvukEOJ3WutMsDDoW3XyIcJOW/S39epwz5CODo5q27L1OMEaO6fyjslMK631AJpOz929xTvcc6u74LvlErKHsS8DXkSIsmlqku2JS+5UbynvQ493B1N5n15g1bhDFNQMfXsSOHEcQpoQay+8KInsG2jNCFp3a613ePzyZk/YO4AeLw9guvZycXgrgZ8IKVdL8a8ppquU0lqrZ4B7gEulEIftYSw/nubUSusXgK8A/8yZmZm5gQLgViHEZ4WUBV5ZrqTWuk+j29Bs9Uby5nEjeXDac94nxudkB7ja+YbsF/sCNf8CmWivsx+O+24ZUPcvmgE9wAcTbHyeWVJZVtQjTBJev/vyWEfkTkLOy5qIWXZyT5iGvA653INmSO2h/smd9p3jR/Jum+3/C2+HSqRgFtBbAAAAAElFTkSuQmCC",
    "warning": "iVBORw0KGgoAAAANSUhEUgAAAGAAAABgCAYAAADimHc4AAAcw0lEQVR42u19a3RcV5Xmt885994qqVSSSpJtWbIl6+W3ZMuWnNhxiInzIAwEQqbTPLrpZprX0MDAQPPoXg0MCTTQTUIPvWbNLJruwKwMPU2HECDvCSEkceJnYjtOLFkPS7YcvVV6VNW995yz54eqiOLOw5IlW4DPWneV1lLVPeeefc7e3/72PvsSFncTAGz27zggbiCiG4jQzIwKADEABGCSCH3MOMTM9wP25wDGXuUel9osmsx+KkB8kkieAAQDeIOLmEh2AuIzANyz7rXoGi3iyTcAVoPEnWBsAyyEUMGS8vKwdMlSURAvtI7rOgCgwzCcGE+K4YEB23+mzzEmdAEBIuxntn8C4PkZ97wkgHOc/MuJ5E+ZTRlAmfq1683q9RtkQbzIkUpKIgLR9PCZAWYLo7WdnJwI255/Xh8/dkSCbYRIjjCbGwE8sRiFQIt08jcRyV8ym6JINC+1Y9duKq+ojDAzgQg6DE06nTJBJkMA4HgRjkaj0nFcOa2FiPv7Tmee/OUjnE5N5RHJcWazC8DBxSYEWmQGlwGUEcm9zKYqLz+WuvqGt4t4YWHEWoupqangZPtx09fbI8aTSeEHvgIA13FNvLDQlFeusKsa1oj8WMwjITA5Pu7/v/vutVOTE1Ei2ctsWgAMZJ/bXhLAq61+Eg8QcJ1STvrat72DCouLIwC4p6szs/+pX8tMJp0zrCxIMgEwbCj3LBEvEjRfdoWurquLAqDk2Jj/0M9+YnUYRgE8xGyvy/ZlswK/JICX1YL4DBF9i9n4O6++zq5cVRNlZvSe7M488ciDDoPlsvxY8P76er1reQUtjUYdQYTBTFo/3neGv9d2XPZNTrgA9PY3XR1U1zXkgYDe7q70rx95QBBJj5k/C9i/XSyqiBYR1l9HpA4ya1W/dkOm9Yor84zWNJ4c8x++9x4R6sB5U/ny9P/ZvVuW5+U72lrSdlqLSCHgCIGBdDp49yMPm0f7TkelkPrqt75dl5QtiQgp7d4nHs+0v3DUI1KaWW/JIqOL7iPIRbAABAAQybuZbU0sXpjZefW1LjNLtlY/9uD9NpWa9NYVJzIPvfU/yLjruCO+T4G10NYitBa+MZgMQ8QdR95cU4Nf9PSGL6Wm3P6+PlNd18CCSC0tX47erq7Q99MRItkE8D/l+r7Yq+9i928A8TEAVwActO64khzHcaRS9tDep8Ox0eFIvuOFd735asSUcieCEK4QkEQQ2UsSwRUCk2GIqFLOXVdfTXHXCycnx72De54MhRDWcRynZcdOAjgAsCPbp7nYcyAXAepZQST+jdk4tQ3rgrUbGyMA6FRvj//cM3tcBuiO7Tv8t1VXR0d8H4547fkSREhpjVXxuCqLRIKfneyWydERVRAv9ItKSp2CeFymplL+yNCAIhI7Ab4LQHLGWH6vdgABYJC4A7DxaF7Mb2ppdYy1ws9kwoNPPiEsWN6wsjrzkbVrI8OZzOtOfq45QmA4k8GfrVnj3Vhd6zMgDj2zR05NTQbWMjVtaXGjefk+YOPTfYMvpi0UFxf1yBsJdBMzh83bLkckGnWkEHx4/z49OTXuFnvR8O+3b5cZYwTNUrIprcV3tl8uSyN5QTqTcp97+mlDBBvNy3M2t14OZg4J9E5A3phVRfL3RQCUXXUFRPgOs+GKldXByppaD8w4c/qU39H2gguAb21p0bWFhe6U1hB07iLIqaKqggL3b7ZtMwD4ZFe719vd7QNAdW2du6yyKmQ2TITvACi4WDtBXDzYKb8EcJVSrt+8bbsCs9Ba60N7niLLVu4sr/A/uHatO/IaqocBWGZY5ldV3koIjGQyeH9Dg7e7YmWGAfHs3j3Cz2RCAHLrZZdLqRwf4CqAvpyFo+J33QjnnJ+NROL7zBYbm1v0iurqCBHh2OHn/JNd7dGIcvSPd+/m0mjUCax9xerPTbwrJfKUgiclGEBoLYjoFUuYAUgial2yxN7Z3s5T6ZTLFv7yFStUJJInmRH2950SRKoVsPcAeCk7Rv6dtgFE4g5m48SLEv6a9Rtcay0lx8aCFw8fcgHgkxsag81lZe5EEEDOmHzLDEWEEs/jMd8PnurvD5/s7w/HfD8o9jxWRLDMr1BFE2GIDYmE95nGphAA2o8ddYeHBn1mS2s2bHDjhcU+s3Fo2iD/TsPQHN3wh0T0WYCDy9/0ZhQWFbsgsvufeFyPjAx5q+JF/p1XXeWE1krMWNGWGREp4RsTfn7vM/4nn94j/uHoEefOtuPirs5Oe3JiPLhiWTnyHEfmdkNOCIExuHzpUrq7uzscTKfcyeS4rq5vICmVyi8o0Cc72plI1AF0HOAjF3IXyAtMeRQQiXuYTcHKVXX++qbNEQDU13Myc+Tgfg8g/t7OK3VzWZk3qfVvVj9n4WXaGH39ffeZe7q7olNhICORPOsoh8fSU86+wQHnkVOnw1tqatiTSlowcgydZkaR54nqWIH+UUeHmJocVwWFhX5xosQpiBfKsZGRIDk2oojEZQD/I4Dgd00F5QzvFwGudBzXb2rZ5jCzCMNQP7v3acFg8ZaVVf47Vq3yRnwf6izVU+A4/Ff79oUHhgYieV5U3/7t21MnTrTZ9hNt9o477kjleVF9aHgw8oV9+3SBo3imKlJEGPV9vL2qyr2xepXPAB3ev09kpg2yaGq9TCnl+gBXAvSFC2mQ5QX0eOuJxJ3MljZs3qpXVld7RIRjzz0b9HR3RCLK0f9y9W4UexFHz1AhDMCTEmdSqfBje/aoQIf07dtv9z/xyU9E4/G4jMfj8rLLLlPFiYR//32/UG0Tk/Tu2lpT6LrSzLjPtO0hai4r5Tvb2u1UJuUKSL+8slJFIhFpjA0HzpwWRLIF4H8BMHIhPGRxoXA/kfgWs43ECgr91VnDO55MBi8eec4BgI+v3xA2lpS4k2HwCtRjmeFJiePJJI1nUjI/P2bf+773OsYYMsYge9F73vNuN1YQt5N+Wh5PJsmT8hU0pyDCZBhiXXHC+dSGjRoA2o4dccdGRwNmpjUbNjr5sbjPbKNE4psXyi8QFwh2XgPQjQCHm1q2keO6iojskQP7bRD6qjIWD/6iqUmNBwHJV8H8BCBjjAJAeXl5RERKSgkhBIQQkFJCSilieflMAPnGvOrMSSEwHgT06cZGVR0v8kMdqCMH9lkGrOt5zqatrQRwCNA7Aey+EB6yuACG1yGSf8dssGT5imDlqhoXzOg/0xf0dJ3wAPBXtmwxpZGIE7zGxGVvxtMBeH4NaEvMr+GUzRxQYAyKPM/56tatFoA91d3p9Z8+FQCMlTW1bll5RcBsQCT/djolZmHjJuICUM3/Ket4+c0trZIBaa01z+19BpatuHxpuf+++np31PehxMJrRCUERn0ft9TWuleWVwQWLA7v3wtrrCEiuallmyISAYAmQHxgoSlrscB8TzERfYnZ2JqGNTpRtsQTROhsbwuGhvo9QcJ8vbUVRCQvZFgqu5Xk11tbSQpphoYGvM72toCIULZkqbuqriFkNpaIvgygaCHtgVhY2EmfB3iZ43r+hs3NDjNTOp0Knz+4XwKgW2rrgzeVl7tJ33+Fx7vg0I8IySDA9mXL3PfVNwQA6NihAzKdTofMTBuatziO4wUAl08/w8LBUrGAMd5aIvFxZmvWNm7mWKzAFUT8wuHn9FRq0o05XvjlLc0ybYwQdOHpeEGEVBjSXzc3y0IvEk5OTbjHjxzWgogLCuLOmsYmy2wNkfgEgNqFEoJYONgpb2O20fyCQn/12nWuZcbY6Ghw4oXnnWm+Z4NuKCpypsIQF00AWqMmHnc/nYWl7ceOOsmx0cAy0+p1G2bAUnnbQqkhsVBphQD+AGDd2NxCrucpAuzRA/tsqEO1IhYPPrVx42vCzgumioRAMgjw5xs2OKvihUGgA3X04AFLgPU8z2nc0kIA6+lnweULAUvFwrCd8hvMhkrKlvlVNTUuA3ip73TQ293pAuAvb9liSqLR14WdF4qgCq1Fseepr2zZagDYnq4Od+ClMwEzo6qm1k2ULvOZDRHJby52G5ALM74LwE6Agk1bW6SQUlpjzJH9e2HBsnXJMv899fXuaCZzQWDnG8LSLE90S22tu33p8sCylUf27wMzGyGk3NTSIgEKAFyRfbZ53QVinmGnR4SvMRuurFoVLlle4RKAro72YHCw3wMJe2tLC9QFhp3nAksFkby1tQWChBno7/N6ujoDELB0eYVbWVWdC19+HUBkPu2BmF/kIz4CcIMQMmjcslURSGQymfDYwQMSAL2jepV/TUWFmzwr0HKxmyTCWBBgV3m5d3NNbcAAHT24X4RBoAkQjVtalBAqALgeEB+dT0QkMH9sZwkR/SWztbWr15riRIlLRNz2/BE9MTnuRpSjv7Jli5hthsOF5OXTxtCXmptlnnJ1MjnqnXjhhZCIUJwocWtXrzXM1hKJLwIozW2cxSCAbKq3/ALAZa4X8ddt2qwYoPHxZNj2/BEFAB9cszbLdi4c7KTzhKVTYYh1iYTz0XXrQgB48cizampqKmBmWr9ps3K9iA/YUkD+ZXYX0MUWQM7pqiOij007XZs4ll/gEsDHnj1o/MB3SqP5wV80NanJMJwz7BRZBsEYCyJia1+2ItZaMDNpaynHzM0Vlk6EIf3Xxka1NC8WpDMp54XnDhki4vz8fHdt42aeds7wUQAN86GKxDw5XV9jNpH8grhfv2ady2AMDQz43SfaXAD8ucYmUxmLORmtZ71kCIC2FhX5+VpJZZLJMXR1dflCCGitobWGEAJdnV3ByOgwSSHNivx8o62dU1++1ijPz3e+sGmTAcCdx190RkeGA2ZGw9p1bqyg0Ge2HpH4+nwYYzEPTtcOADfnnC7Piyi21h45sA/GGrm6KBF8aN1aJ+n7mMvqF0RIGYO1xcWqpWxJaKwRH/7gh6inpyfjOI5xHMf29vb6H/7Qh2CMEVvKloTriopUypg5qTopBMZ8Hx9YvdpdV1wShCZURw/stwCs47rqZeeMbgJw5fnCUnGe6I2I5Denna5yv6qm1gUYfb09/pnTvS4A+5UtW2yB46iQec5LxU7/Vn6ztZUcIcO9+/dFNq7boG649np9w3XXhxvXb5RP730mIknob7W2kiSSlnnOW1pbi3zHkf8tFzM42eW9dPp0AABVNbVu6ZLlfjZm8M3z1SLyPH5nAXELEX0KQLD9ql1UUFjoGh3qPY/9Eun0lLOzvML/m23b3GQQiPOBnYIIGWNQX1ioWpYsCZ8aGDD9k+POic4OdaKjQ/mBT1XxwuCHu95s3lxR4SXDkM63v7TWaEwk5BMv9QddE+POZDKpV9U3kBBCFhTGbWd7myWiKoDaAD4811QWeR5gI5pNKy9aUV3rr2vcFCESdOL4i35H2zFPkLQ/vOoqWxmLuf4c1cG/mxRjsL64WL23ro7XJ0qCqnghb1u6zHx47drwGy3bZFNJiTcWBDQfPoadTgaghqIi84P2dpqcmnAK4oV+orTUyY/FRHIs6SdHhxwisQXg7wEIL9QOyK3+zxLhZimlv+PNu2UkkudkMunw6ccepTAM1C119f6nGxsjo0FAap5gZ04InpRyS1mZc31lpbyuslI2lpQ4IMjUjFyi+eprTWGhbE+O+4dHhp3k8DBX1TVYqZRKJEq4o+24ZralAPkAPzaXXSDmCDsriOhzzNbWr91oiooTLhG47egRPTU14eY7rv7r5maRNoYWJOBAL3MfORiyEL5Fzjn7q+ZmGXM8PTE57rYdPaIFEceLitzV6xrttHNGnwWwYi6wVM5JACT/Hmy3RaL5me1X7XKUUnJyYjx45te/ksYa+V82Nvrvqa/3kr4/r3SzYUaR62IgnQn+94l2/3+9+CJ+1tNje6cm/ZWxGC+NRmVqHtTdjEA/fGOwsqBATARB8MRLZ5zk8DBW1NRo1/VUcUkJujtOhGEY5INkKcA/mW0ukZwD7NxCJL4LWLt52+V62fKKCIjsoaf36KHBfq88Pxbc+aarJAOvyO2cp8nnu7u7/ZsefpB+3Hki8uzQoHx2aFA+0Nvj3NXRYapiBUFzaalKG0PzKQRtLbWUleGujk4zmkm5oR+EldU1ynEcqVxPn+7pIiKxCeD7AZyajSqSs+V8iMT/ZLZrihKl6ZbtV3hEJIYGBvxDzzzlMjN9Y9tl4a6KCm9iHgk3w4y44+KxM33+2x+4X02FgbNieWXwrpveZZo3bzZDg0PmzMiw++OuTtqxbFm4prhYZeZpJ+RiBmV5eTKmVPDznpNqfHRULF1eEeTlx1RRcUL09fb66dSkSyQqsufO6FwFIGaXYoLVAL0FgN2wqVko5ShmNof374WxRjaVlPl/snq1OzbPKSaCCJqt+cK+fQSwumL7jszh54/w9//5+84//tP3nSPPH+WdV+zMAKw+v28fQmvNfNqEXCrLHzU0uM2lS33DVh7etxfWWiOllBs2N2fnh64DsHY2tmA2AoAQ6h3MRuXHCjPlFZUOCDjd2xO+dOaUCyJ7a0sLPCml4flLp7TMyFMKL4yO6kNDQ0qQsH93+7dRVFTk+b5Pvu9TYWGhd/vtt0OQsM8NDznPj47qPKVg53kcrhDyttatIBK2v7/PO9XbE4IIS5dXOLF4UYbZSCHUO2czt2IWsBjMuAoAyitXWNeLSGuM7Tp+HADENRUrgutXrHDH5jnFhLNRq9OpKcdYLfPzY6aurk5Ya+G6LlzXhbUWtXW1IhYrsNYa2Tc1pRTRvGbV5mIG11RUutdWrgwAUPfx47DGWNfz1PIVKzk73qtmztl8CCBXWcQBeDUAJMrKpJCCMum0GR7sVwDwR/X1kESCF4Bitswodj1NJGw6nRJDQ0NMRAiCAEEQgIgwPDzMqXSKQMIWu66x50F9vEHkTPxxfR0A8PBgv0ynUkYIgZLSMpktXrQagHeudPVsFHURgAQA5OfHmIiQTqVMKp0iSdI0JhLkGzPvUf4cGbc+kRBV8UJtjJa33XqbJSLteR48zwMR6dtuvc0aHcqqgrhen0iItDFzZaVfd7J8Y7AxkSAlpE1n0iKTThkiQl5+jLNSSgAoPmf7Mpt4LzO7ANhxlAUAo0MATBGlbNx1sRCrDlkUUuJ56otNTekPPf6Y/cEPfxAZ6O/P3PSumyxA+Mndd5v7H3wgAsB+vqnJJDzPHT7rkMd87UbDjALXpahyMBFkKAw1AYBUKqul2cHLtereEA2pc02lBBAQQTODtDYCAKRUDBD7RiN3lnchspdymQt/2tDgdU9M+F87dNB94KEHow889OAr0OrnmjYHf7Z6tTe6AJM/QwUhpTVntAZArKZPQ8Ga31S+0TN4IZ6PHZBrYwCNAihIp6aImRGJRlXEi3DGT8sXx8b0uuJiTGkNLFAm24TW4qtbt3pXLV8e/KC9PTgyOqoAYENxkf7junraPc2EClqgkCcDcIVAezLJodUi4kZNJJqnGIx0aoqyh3DGmH9TMnNeVFAu+BwAdALAypHhIWOtRSQ/XyZKS4O+073qR52dfHNNDVtmkgsY8x0NArFr+fLINRUVdiKcXmgFjiM1sxhd4GwLywwpBP9rZycDoERZqY7m57vWWowMDdvsIDvASJ9rLaJZ+QEA/xoAzpzqpTAIjJRSVNc1MAC+t7vLfbyvzy+NRBBYu+CZzcO+LwyzMMxi2PfFQqe6BNN2CHv7+/0fd01n+FXXNbCUUugwNGdO9+aW65ML4QdkU+rNT4kkj4+NRoYG+kMwY8WqGieRKPNDo+UHHv8VnZ6aCkojEYTWwuRKCczzJWbA05zhF9nTM/N5WWYY5hwIwGAmE/zpr34FX4cyUVLmr1hV4zAzhgcHwrGRIY9IgNn85Fz1/2y4oJwaOkMkdgO2eiqV8mvqVksiqMSSJaa7vY2H0yn3F729emtpmV5TXCwiUlKuoJKSEkqI356LCK6UiDkOYo5jDw4O+jc9/BAfGx2OKKnCK695C0eieS4R7L4nnwgmxsciROIZgL86m1JoNIfcz2uJ8CCzCS/buSusXbM2zxqL06d6Mk89+rDSOlRSSP2HtbXhO6tXoa6wUDhCeIu5TO9rwb7QWr8jmbT3nOzGjzo6nNBopZQTXnH1taa8ojIipERXe1v6qccekUTSZcbbAPPz2RQEpDkm4P5fIv6PUqn07re+nRKlZRG2jJGhwcz+Jx/H8PCgl8PAgqR1hZD4LWyBtcayyWk8Lild4rfsuBLFJaUREoTkyIj/0M/vsToMo8y4F7A3zrYaI80xiyJBJPcz26pINJredf1bqaS0LJI9sxv2dnXqno4TGB0ZUql0WgD8WykAgExeNGqLS8t0VU0dKqtXOVJKJaTE2PCw/8sHf2FTU5NRInma2WwBMDgbHmiuWiGn35qI5KPMJuE4brplx5VcXVfvUTYQY40x6XTa+OkUh2Ho/RbOPjuOE3jRPIpEIkoqJXja4JuTnR3B3id+hSDwo0QyyWx2A9g/lzKYdJ5JWc3ZcpNVAIfLV1QH6xqbREnZEkc5jhJCgIjw29yYGdZa6DDUI4MD4QtHD9tTJ7scgFwi0ctsbgawd66FYGkeMuPKAfFdIrqJ2QCALkqUZpYur0BxokRE8/KsEEK9Hijj3DjowlpZej2oSIC1VqdTKTE2OmxfOn2axkaGvGlmRIKZfwrYP8fLIUhzkesNyXcQyV8D8hxesvCql8lu3wtxmbmNUTKRfDJ7UmZeCp7Mx5qjV1YjV61EfAOIdoC5FkDh61AeObYwCrBDJKyUygBMC1ClhABiY7RktgKgEED6DRhLDSAJog4w72Gm+wD99FnnInix1I5+tYrkTrYioXodLkoTqf/OrP+gorLKb2ppFWEQ0HzbDmaG47p85MD+sLenK49I/Ruz/s+5MbyOACZeJett3lSOmsdnNDMGl9sRIabr7rxRS2U5dYpEo85CGG9mhut5kErpGX0OzAL5iRmqct70vcL8NzOLHZZbfSI3ScYY2LMKLc2XAIwxM6ut5Bys19sBM2PiC8IwLoQAXs2rf73/88yKVtOTNf9VAabv+XIfZ/V/0V7koLAoGmsA1g/8UCpHeEQL4jkLKU0qlc6uZg4Ww5OrReLsPAqIDwyc6aMDTz2RicXjwmbPe52vKsqpHCEEjyeTdqC/TwJCMPPDs6GNf5ebyL6s7d6zXtRm5+hPvNo1416CAbo7CxbEpVeYvIzDFSA+TiSuA3E9GDVKObooUaznmmxBBB4bHVU6DBUInWA6zmzvB+w/zMjb4Usv8Tl7Swj1EWv1/yguXTr+1ptudowx7pwcE6XC+37y43Bk4KUCIdTHrdXfXWzPqrC4XirnYbpqrZNVzxQGgWOtnZNRZmsBy3qGUyizn/5i0f1qkQWi9NnetDiPLOuzfptzomgxGV4FLN6oIFuGDsPAWMMg4lnCH2KrNOM3nteiRDuLUgDWWgcQnBwbde/91x8Jmj5qQ7MVIYNVGAQSEDx9z0sCOFem/iTAxNZ6fiaFOa5gemUUlboX405YrOEqAnAZkfxnZtuwsqYus6KqinQYvqFjxsxQjoNTJ7vpZFeHSxAnmM37AexZjGposdoAZCdsEOCGeGGhKK9c4Qa+f04CcD0PybExA2aAMAzgqcW62NQif6W5AgCtNcIgQJg9jPFGAiAimJeThNWMWMUlAWB2tDbn+KCZ17kcLZ3xvZnwc1HyMJfaJQFcEsCldkkAlwRwqV1CQa+JLX9zaOJcImDn+t1LAjhXh0AqOK4LtnxOfoDjurljo5cEcP7cEGE8OYYzp3qtDsNzgPPTVMREMplLOf29j/li7hyV2J+NE+s5xIH19G/Fs4uZ91KLGBwYgH8JUBMgOKt66NyTITgb2OFHzyq5s6ja/wejb9Sz9Tcp8QAAAABJRU5ErkJggg==",
    "question": "iVBORw0KGgoAAAANSUhEUgAAAGAAAABgCAYAAADimHc4AAAmpElEQVR42uWdd5xdZZnHv897yi3TayaVSSaVJBBIACkBgiF0QRYEBFwsK7uo6OK6ltVVLOta1gaKYAMFQUXRRUCaEKSXQBJIIz2ZlEmmz9x2znnf/eOce3NnMjOZVIJ78jmfmdy599xznud9+u95XuH/xyGAiX6/GviwUnK0QGn0t4N1GIPJas1K4C7gJiADKEDz/4j4CogDvxbBKCUmYsghOUXCE3gOGBPdl+Igc/9wOfKr7dfAFYAPqMZxZaq2Oo7W5gCTwYAIxk+jJKA3bVixJmOAQATbGJYAJwFpwPy9M8ACAmCBCA8L4iUStnPzN07hkgsnknDVwVmBYmFyrfg9bxJo4ZlXevinz65jQ3MupxSu1nwO+DpgyyFUAbJriRT08VCfKb43PczPDcaAuy1LLgsCE/zwG3Pt6z45m3RzT/QNMqxVbQyYSFjUHj9nQGyCnhUE2XYq6uMsXNjJmVev1H5gMIYVwFGAVgeR6HZEeBMRwY/OICLoUGfx+/0iBljROdyFEwCIcGQQGKmscNUlF0wgtz2FAUQEEYY8tQ6JH4tZlCRtEnELESEIQhs6+GcFZZeilKGjxeOUOaUcPS0pxqAsJY1AHWDsg7Ta8wQEGAXMAmYCTUAtkMwboUGIlgHagY3AKuCN6Ge66H12EcOG8nyUIDGDoazUxXGsiHCyJ/cFYwwlVTHIBmxq7qGjK0cyYTN2dCmlNQmyXVmCwKCUDKqKJLwBlEBVhZW/MydyCrAPgr4NgBHAxcDFInK8IOWmIMrD1yLhCpXwM4b1BvM08EfgEaC76Hv1cNTTcL/ahMuUWNzm1/es5LY7lrF0WSu9KR/XVUwcX8GVl0ziXz4wg1jcJpvxB2dC0XIY6PvtA7Tq8yt3NPAJ4B9FpM6YkOAGU7xSxVLWkEpEa40JD4wxEhG5UUQagauMMRuBO4HbgA39FsD+Ou6IEhC46trHueveVX3+ns0FvLp0J68u3cnv71/L724/ixF1CXKZYM9M4MAzoPihPwZ8oYjwAUBlRZUaOWKkqqutUxXllcRjcSzLGlQFGGPQOiCby9HT20NHRzs7drbQ0rpDZzLp/BoaJyKfM8Z8BLgZ+DbQsTfSMCgDtCFR5vKhj/yVu+5dhW0p/EBz5JQqmpoq2b6tlxcXtaBEeO7l7Vx89V944v4LsZSg98GhtQ8A8UcCPxWRcyMC+rZlWxMnTLImT5zCiLoRxNxYuLKNxmiD2QN9hFD1iAp/6kDTm+pVW1u2snrtKjZs3GACHQRAhYj8hzHmcuCTwJ+K7NBeMyEIDMlyl8ce38TPfr0Cx1YoS/jRt07jH987Fde1MIHhsac284/XPc6O1gwvvtrCLT99nU/eMJuetjS2pQ4JA/LEnwXcJyKNxhgfsCY1TbZnHz2H2upatNZ4vkc6my4Qdu9VgkFEiMfiTJowiUnjJ7Gzbae8vnyJvXzVcqO1DkSkyRjzR+C7wKeie9trlWQwiC384u4ViIDna775HyfyTx87htSWHrxMgDGGM88bzx03ncE5VzyAEuGO367kox+eudfEZwhPZDjEnwM8nid+IpG0zzrjbFkw72wqKypJZ9LkvFz4JaJQogpGdW9OpcLPaaPJZrNkc1mqKquYd8o7ufiCS2TMqDG2MUaHkab8K/AgUBXd4149n20psp05lixrRSmhuirO1ZdOJteSCh/cEmxb0bs1xfx5YzhmZi3aGNZt6Ka5uSeUEGMOKgPyLuYk4EERqTbGBLXVtfbF5/8Dk5omk8mk8X2/QLgD6uNGTPF9n3QmTV1NHRecdSHvmHOiEhHLGOOJyALgscgT08N9RhNmEAi0oavbIwgM1ZUxKitdtDahYc6/V0AsxeiGkoJh7k35KDV8T2tfGJAPgkqA30fGNqivG2G965yLKC8rJ51OHxTCD8QIpRSe5+H5HscdczznLbiAWCzmREw4FngIqBguE0RCGxAvcfjqZ4/j9JNG8eVPH4cbt0MGREzS2qAEvIzPyjUdAJQkHaoqYgSBYW8fXQ3TzcyrHQ/4jojMNMb4FeUV1rnzz8N1XXJeDqUUh/LIS0QqnaJxbCPvOvsikolkngnHAL+N7t8fFjGUkEv5XH35FJ743wu54pJJ5FJ+n787jiI5qpSf3rmcFW92ICLMmFrNqNGlZLPBXi8+NYxVb4rUztdE5MNAYFu2febpC0gmk3ieh5JDS/y+hFOkM2nq6+qLJcEHFgDfAnr3JqpJ9XikesMTIJl0CqqmpS3DN778PJ/4/LM4jsIYw0c/OAMV/X4g44D8qq8HvgRcI6IShCGSNfuYOYxsGEUqlTrkK38wJmQyGUbUjWD+aQt48NE/24A2xvwrkNsblZsPqLQ2uEmbBx/ZwGe+8jy9aZ+29iwdnVmUErQ2XH3pZC77h0mkO7NYB9ALsiPizwVeFJF/ARLGaN8YI+PHTeDo6bPIZDKHBfH7S8KExiaOO+Z4jDFKRAzg7mtUrByLm3/+OkuXt7F2fRcdndkCc669Zjo/+cE8Mmkf9tHu2YOsfB84C/iTiMSMMb7jONb0KTPsSU2TqamuQWvdx1fPB1f5IOotZUI6xexZc9jYvJFt27eKshQE+xgca4NtCUoJliVMnlDJCbPrufKSSZwxbxyZ3hxGm31+ZnsQN3MacG9E/KChvsGeN/ed1FbX4vs+nu8VvtAYg2VZOLYTJpy0IeflCgHUW3mcfPzJ3PfAH/ZJN/eXBK0NlhJ+9/MzmTZnBHR7pLqyBUdgXw97kNduF5FSY4w/dvRY+5z552FZFql0qs8XGmNwXZfu7i7WbVxHJpOhurqGxrGNKKUIguAtYYKIkMvlGDliFBMnTGLV6pVFXtO+XK9vuiJoz5JK+zj2/qtfu9/vPnCViBwfuZn2mfPOCh+on5uptSYei7OpeSOPLnyEdHpXqn7MqLGcdcbZ2Lb9ljLB1wFHT5/FmrWrCXRAJuPjeRqV3LschdZhFCwSGl7LEiwlB6xgXVwIUVE62QBy4nEnkYwnw8i2yM00xuA4Dm0drTz817/0IT4Im7ds4qHHHyyop4FUQP61PHO00QeeAZ5HXW0do0eFQIQdrWmeeXErdtKJivHDCT+FeMwiCAw5LyCZcDDacCARAxRlD2eKyCxjDLU1dVbjuPFkstndPJ1wJWgeffJRsrnQKzj9yIn89KNXURp3QYQtW5t56LEH0cZg2/ZuRtt13UI0q5RFIpYoXHd/dXaf5JoIkyZMKrz2xwfXg5I9ZmTzqsfP+Hz+htm8+9zxfOcrJ9HYWE4mu2+5/6FUUB66cbIgYjD++HHjbdu2Q4NbJCjaaBLxBM+88DQ7W3cgIjTWVfObT36A+rEjGVtZzru+cRs5P2BT80bu/8ufWDDvLMrLyslmsxhjiMfjrFm3mkVLFpFK9VJaUsrkiVNpamwimUwSBAG+76ON3i+vSonC931GNYwmFnPJZnM8/cI2etsyOLaFNmbI/KxSgpcJOOrIav5w7wWQC0h35Q6oSu1vRabmV0ZdbR1a6z4pZGMMruOyfcd2lryxGCWCEuH2666kvrqSrubtLDjuKO771IeIOyFvt2/fyh/uv5eVb65AG43ruKxavZK/PP4Q21u20d3TzdbtW1n4zBP87k+/4W/PPUXLju0opUjEEziOWyhLahNKx95ISBAElJaWUlNVC8D6Td28ua6TWMwaliqRqJT4xivbaWtJ4bjqgElosQTkr1iTv3g8nhj0i1559aWCqrj+7LmcOns66fYuEjGXdFc355xwNA/9x3Vc9p2fsb2zh97eHh598hEqyitwHZedrTtCL8kYRldX0NzWCUB3TzeLX3+NpcsWU1tTzxFjjmD0qDHUVNUQj8URFaqo/GmMLmQf825v/9VpCO1QXU09W7ZtIQg0y1e1M+vYekzvMAo0FS63/GQpN3zhWU44dgT3330OMVuh9T7HXkO6obLbL/0epKenh3Ub12GMobq0hP+89Bz83nQhDLcti3RXD6cdNYWnv/ZJ/uW2e3hsSegGdnZ17no4rblg9gzu/MQ1PL9yLT9+5GkeXryCVDaH1oaWHdtp2bGdl197kfLScmpq6hhRN4LqqhrKy8pJxBORHbEK7uWgMYiBqsqqwn9Xr+0M7YDZMypOlOK3f1xDNhvw1HNbWL++i6Om15BK+QdEFdn96J0rJlCx+hHC1ZdIJJl15Ew6dzTz5cvOo6aqgkxvCqvIUNuWRaYnxcQRNTz6hY9wzzOL+NXCF1m8YQvpnMcRdVVcOXcO1597OpYIC46dzoLZ03lz01Z+//xi7ntpCS+v2RStcujs7qKzu4u169cA4DoO8VicRCJJIpEkHouTTCapr61nzKixuG6MXC4KkgiLOSUlpYX7a97aG/l5Mowasebq90xm0ZIdnHRcAxPGV+xT1nO4EpAp6Nsg2G1xKBF6PZ8vXXEhZzdV4toOuVS6D/EL3pulyOY8BLj89BO4fO4cOjp7yPo+tWUlWMk4Xm8a3xi0F2YdJ42s5zOXncenL5rPa+ubeeL1VTy2dBWL1m1ie0d34do5zyPneXT1dO/2vVWV1bxjzolMaJxANpstqKd4LIZSoU/f1pGFPRjgfAUs2+PxwfdO5Z1zR1NXk8B1LXxP77X6MSaPqBuaAdldErB7qBJoTcyC2fVJXMsik/WGDEhUdJeZnl4EoTwZR4ngBQG57l6sqHhjRe/L5jx0JoelFMdMGMsxUydww0Xz2dnexfLN23htfTOvb9rK8ubtbG7tYGd3L6lsjqDIxW3vaOOhxx7g9FPOYPrU6WQyGTBgWTZKFBpNJuvvlS+fyviMG12K5+u9I74JCpBGrWFnu5/nRo4IZLabBOwyQH1VkBLBE5tjG8oYXV1KNpMbdjSYlxA/CApxhD2A1CgRlBUxI5tDp7MoEWpKksydOYW5s44MFbrnk0pnaOtJ0dGbpj2VZm1LG7c98jTPrlqHUoqnnn2SuppaaqprdzkTssu9lEHSCHkoomX1ffZsLogWIVhqTymNEJRn/G5AqBjh8Oij7SxZkTZKMIFmHbADkEElQOu+KkhEePmV51lvZbmg8RJGVVeh/b0zRHvz3mJmeEGAToWIhBAcK8RsmzE1lYypqwYR5s6yueLEY7jse7fzx5eWALBo8SLOeufZBBHj875ee0eWrrYMrh2io/O3ZYwJiy+uIt2Z3e1+LUuRqHTRKQ/P14M8Twhh1blWCLowWPzp/jau+/x6/MD4SnCB26O4yx5cAiKx1kYTd+Js3rqZJUtfAeD7f67lf669gnRnN7YlhySvY/VDJAfGEHh+SFMDOp1Gofjqe87jbyvW0Nrdy5ZtzfT29lCSLI2eKeTAU89tNVOO+7WpqYkLOqzGR2uW+tqkXHXJJN5/5TQyGR+JpMJ2LTIZn6/e+ByPPLnZ9PTkBjHioWdlggyCpjdtzPrNWQNoEVxteAn4YT7zPLgNCIJQBRlQlqJlx/awGC5CQ1XF/oDPDhgKOE84BCyxSWVyNFSWc3zTETz02jIy2QwdnZ2Ul1X0j2lkW0tKtkVwk+LjDdp54ulmVqzu4JtfOYnejiyWo9DGcNFVD/HEM1v2tqMjLyiWMTwFXFbUprQbOjorhHmSgthGotnW3oYxhsAY5jSNBT9ADrP2DlGCbSlmjBnJQ68twxhDe0cbjeMa8wsmT7w2EdkpQvkAmq8e4Fs3v8ZlFzZx1MxanITNP3/sCZ54ZotxXUt8Xxtg254ReAYMWW1YBfwmUj1BdA96SC9I66AA/9Ba09UdBlHJmMv4uhrwg4KXc3gcBiWCrzWTRtYVXu3o6iiGyuio4rfQGHOJMZQWgYbzuNIv2rb6pIgJFj63xZo9v5Eff/slbv3lMmPbSudygQHOB54u+sxQh1dM134Ng7sxIB+Iie/7+L6PH/h4nkcqFcbt9RWl1JWXhBIyTAaEeRwzYLyw71WqfteMgFW+1oyprizkj7q6u8jmsn0kGnAiwnUNQJhF+fbG0qTDqle28YnPP4OlJAgCbQPXAw/vQ84tX200g8UBAnTn8/JL3ljMilXL0FHyK50J9WV9eSnJeAzPG54HZIzBtW1wbLxM9oARf6BrCmHnSnVpkrhjk855bG7eyJ2/vaN//j/bD3xQXJBK5O3F1u0prvrQI2Szga+U2MZwJ2GrqTNcrFFRi5XeUzLOEHavoJSQzqRJZ9K7+fI1pSWIbaNzXiGAGopQjuuweWc761paOWnK+CibuX/EdxyH5tZ21mxv5eQp4/vk9n2jKUvEKY3HSOc8/CCgq7s7bw1FRIwxZipQXtTkUdxWGgRRAf+bN79GKu0HIthamyXAtdFK9g+UF6KKagETga+LiNHaqJhtMWX0CI4c00BjXXXhA5UlSRiGKslXw9q6U5zx5Zs59Ys/4H/ufwI7Ee8Tue4t8S2l6OhNM/8rP+K0L/2Ab/7v49E1Q5WktSFmWyTdEIni2jZTR4+gsb4GY4yKgLzTgTuKjDIDtD6ZVNrXIcfojryXVH8dfiByQXmL/J9KSbXWxj9jxiT7R9deztiqChLxGM8sW83c//w+ACWxCP2w525ZLNti445WVm/dgRLhmZVr9+vWDWA5Nps2b2PV1rBJ4umVawt5nUCHDLAtq1CPqC8v5dmvfIJ4PMZDi97gQz++2+roTQfARcaYdwKPFyEAATZHi1JEkEgdfRBYUaSmDmhBJgBKRGSBMcbUlpVYd3/8GqaMHYkYEMvqoz9DVSTDJqSKWsR1v3TAft20SLjaC7osvKjvB5jIG8qXUYPIprkiXDz/JL5+xfmYELFlCPvY8hfI18QfB34R3XIa+Bzwu4NB/OKKWK0SqTIGOfqI0dRXV5DuTkU5dt3H3cwFkSGX4RTFfSY01HHKlAnEHZt3zZm5XxKQv2ZjQw2nTWsi7thcOGdmQTpyfhChmE2hyK9EQcSwoLOH046ciFIigdYiMLZfQUpHbuMHIjV1ZNRQrQ4G8fvXhAUo1ErzRBdjKI27WEoItKGjJw16z6lciaArJTGXhz73z2zv7KZpZB1eOrPP7mj+mknH5YHPXsu2jvCafjqDMZDzfCylSHseqWwuUpkuCccO1Vf0TEoEHdJcDfFVK/vVyzlYcxTybpkP0NGbikRcExiD9gOqSpIkY6FRa27vxAwTDZ1vpki4Dk0j68jlvCEQgAY/0H3OQOvdBCZ/zbiz65oiQjbn42uNbYVGuq03dJurS5O4jkPO89FKyHp+WGwSjOkbIPU3N+pQTDXJr/yWQOvNSkQvWrfZLFyykkRdNclEDOU6HDGyjnG1YUlvfUsrrV29OLbaozsZdp2EkpPNeQPGDaHnoonFXBLlpbvOilLiyURkXPVuTCi+pjaG3mwOE3k9G3a20ZsJJWDmuJGoijJKk3FUMsGtjz2LMRgVtrwsZXDLpA/FSBm7CIx7h4Gv5vwg9w/f+bn7+UvOZs6EsehAE0/EqC4J23Faunp4fdNWTj96KtpLYw0gCeEKC3P+Yf5o4NStMYaY64BlsWzdZhYuW82KLS1kfZ9xNZWcOGU8p0xtIl6SIJfK9LmGSBh4KRFSWY+sF2YubaV4Ze2mwvvijsOLr6+ks7OHe55+mZ8/8YIWEQm09gln+MBbOLtHis448FfghIFC5qiJTfmBVp++cD7//cFLSHf27NYZaIzBLUmC79PTm8a1LdzSJGQ9srld8EZjDMqx2djSxqfv/BP3v/I6WW93O3fs+DHcePl5nHfsDHK53ID5p5aOHjw/DGgTrssF37qVl9ZsRCkJtO6XBgU7euFThP3FB6TBe3+73PPBRQ3wExF5d97V69O+H73WNKKWpd/5LI7sak7OAwzcmMtdT77AL554gTXbd1ISczl1WhP/duF8JoyqJ5sK+8gCrYmXJDnri9/nkSUr91BREzbe8mVGVpbjFRWBlAidqQydvWEZI+m6LN20hXO+fgtB2GkvBVBuQXXpLsKGk+++1cTvn4oQoBW42BgzNwiDlFFFdiLQxpxhKTVhzfad5n9fWqouO/0E0t292FaYL48n4nz01nv44cN/65tj37yNe19YzP2fuZYTpownk95l+45uHMMjS1YyrraKK089jrlTJxB3HN7YvJW7n36FZ1eu46gjRlMSj0W2YBfxM55PdypbyILGXZtf/e0lfK2NpZQExvwNWG6McQxkMGYZcD/heAP1VhN/IOMj/fzi/sf5lpL7tTbBzCNGWS9/898xgcYPApIVZdzyp8e57me/w7Utcn7A+IZ6Wju76EqHK3TCiBpe+/ZnSdgWWuswTQy8uHoDTQ211DfUgR+E4mTb6FSaF1dvYGJDLTXlZXieVwBfBYGmpbOHIAibQxKOw9qWVs782g9N2vMMkDbGjI9qr0ONWOCtnijFEC2pqui0gNXGcL6l1OhtHV1BTUlCnXz0VBTw+trNvOe7vwBC1MO3PngFd3zmY7z3tBN5+o1VbG/vpK0nxezxo5kx8Qi8XIixNMARI+tIOjaZ3gyB5+F7AV4miwgc0VBHwrYLqiec4QOtXb34QWjcA2Moi8e44Vf3sax5W2ApZWljfgz8HogVPZN6q43ucHvE+g9Myg9N8oFPa2NQSswXfvsgq9Y3Y9k277vpl4Xs49VnnMy/XXUJOpulqamRG959diG4W7GlJUKlFeWG01k8P8C2FJZSWFFlq/C3qMcgj8/f2dVLNioIeUFAXVkpdz39Mg+8+oa2lKhA63bgvyKJ9vo9z2E1rXBvZkXk5y88Zoz5pULe153O+v/047vtGWNGsnjDFkSEiSPruekj7yfX0xPmkIKARavXF3z+8kR80K7EoToWlQieH9DanSLnB1gieIGmMplg0frNfO4396NEtDbYwL8D2w8nVXOghnXku86vD4w5RYQJTy1fo59avkY5loUBfnHDtVSUldLT1U1peRnPLlrCTfc/WgBgzZ85GZP1ho2vz3svqWyO9p502KESEb8sEWNLeyfvv+VOutNZT4k4xpg/AD99OxB/X2ZF5BVHJ3CpMWxTIri2bbwg4MYrL+bkY4+ip7sHNxajuzfFB773EzzfxxjDx889jWlN48hmskPWk01R1jMwhrbuFK1dqQhBIuSCgIpkgi3tXVz2/dtZv6PNt5Q4OvRyPvB2Goy6L+NqdPS5RcAKEWnI+X4wf9YM67NXXkymqxtLKdxkguu/fQsrN2+NUgKj+PJ7LyDbmy5kJ4da8doYetJZulKhDQgj7jAFXV9eystrN/GBH/+aDTvbfEspO9C6GXhXtDjUW46bOYgMyOfFP6VETtfGBLXlZdbPbvgwOggIAk1JWQl/fup5bn3oryglVCTi/OZfryFZVgLpzNAVNW3IZD16M1lyvibu2JRHVTQvCIg7Dnc89SKfu+fP9GSyvqXEDrTeBJwNrHm7qJ59ZUA+bzQH+JpSov1Aq1s/+n7GjW4g1dkd6nal+NkjCwtpZ6UUn7j9D2Qi3T/U0vSjuCLfmqSUMKqynAvnHMXU0SP42n2PcN9Liw0QKBE70GZxVFhZ+3Yj/t4yIJ+uKAF+ZVvK8QOtrztvvlw8fy7pji5sy8IPwtaRI+pr8mlfWrt7eWTxiv260d+98BolMZfebC4/f8DWxtxDWCjvejsSf28LhHnV81Ol1Ae11v5R48fZz3/vy6FBjTKgxhgs26a1q5v3ffsWnlv+Zn54516jIUQEPwgKxRXAWGEeqS0qFd52KIomhwMD8qvrchG5W4n4jm3Zz33nRmZNGk+6X5NGCB2xUQibdrYWum32jv4hfEXZFr978jk+c/tvTATG6gTeEVWs9ntK4tuBAfnV1Qi8altWuR8E8oNrr5aPXXYhqY5ObMsatBE75jjDmIcxeOejMQa7rIR513/BPLl0uVhKZQKtJwFbI8K/refw28OsFyjgl7ZSlX4QBBe9Y7b62CXnk+7sGpD4xZDgrOcNHV5rTTIWA9cZpK9HQSZHZypdQCAWZTIVb/PDHiZs5UtKZK6vtT+6ttq+9eMfws8Or2FZ9hBwJUuSLF+3iQ0tO3azEXlQ/V2PPcWra9YHImJprV9nF3ZH/z0zIE/8KcAXAG0pZf3s4x+ivraGdHc3lmXt8xdrrYknE3z33gf47O2/2ZOkGMCO1NQXilDO/D3sLrGnv823LaW0Mfqfz32nnHXKCaT2k/j5WREbtmzn339+95DEt5QyllIG2BmlGR46XIopB1sC8grh9Hxd9dK5x2MGQTfsLQOUbdHc1h4GXSJMbKhjwcwpZD0fI5BwHJ5ZuZbXNjQHEt7nRwinIMYjGI28nb2fPTEgD9VLiMg7tDFSV1GuZowbgx6kML6vEMM8km3OhHHc/P5Lae3pxRhDbVkpNz28kOtv/72ocCLqRREDMgOAiw8ZjORQqaD86zOVyBjAHDdpvKqpqSLn+wd0Wkh+CWc9j5aubrrSGZQl9GQznD9nBqXxmBVobZTIFYSI5lMJdyJyiopE+UKLvN1sg9pDfHBqROzg1JnTwLY4EINCQiC+0FvUXJF0HWylKE/GqCpJoJQwfkwDt334chzLEsLtp95nWWqhUrJChGUIjwDfB64BJhfNOJW3yw5R9hApZ4B5ESpN5k6fDJ5/QBrzgiBAYi6Pv/ZG4bXx9TUhgFaHkMRkaRK0Jut5IeoiMBIEOoh2fioh7GeYKErOjEbK5Iw2TwE/Au4bqB/r7cKAfL9AuRI5ThvDmNpqNbNxLH4uNyQmNA8z3NNRWl3Jm2+u5daHHocIvXb+sTOIOw4lCRcUPL54Bd/642M8vHg5ylLoQNM0c5Q1ftoI1izfbHZu7TA97Z4xQb59G1eUzMcwH3i0qir+gba2dPPhzgR7CP//GFFSR2D0CVOaVFlFOeme3kGRzVpr4vEYxNzBpT8amvDc4mVc8+1baOsOG/+OaxpHTybLnU+/xJvbdrBw+eo+8EIdaE678Gg+8t8XUVGdpKc7Ja1tO2Rj8wa2btjJhiWdvP7kFrNtbVd+B6Yz29rSF3/RfPHmG+XGgwYtP5gSAHBa9Is+beY0hVJD5mviySRLV6/jwZdfI5XJ9enbyo+Gz/o+i9dt5NFFS8NOFjsEdL24dgPv/NrNA67ThnHVXP7xecy/bDa5tEfHzh4sy6K+ehT1dSPonLGN7gU7OPvaabL4sWbrL7cuM8YQzL9yWs2NcqOe/eHZ1iu3vXLYSoEMkXz7q8A8EQle+v5XrGMnTyCTzu5WTA8hhgkeePYV3vNfPyCdy+33TTmOxYQZozj9olmc9u6jqaorpbsjXdifqzAhC0Gw6Ehvpy23mWRZjO62DDrQuqq+RLw2/dHrT/7tj75ovqhulBs5HN1UGaToUqNE1mhjKppG1pulP/pvsa0hdodQilnXfYblm7Zgi4VvgiHcLqF6bAkz541Ce5rOnVl8zyeWsBkzZixHTG5g4sxRjJs8gljSJd2dwfOCIQZjGyzl0J7eRkv3BhzXiXbL0iYWd1VtdcW5V4z9ycOiRBttDjtJGGx08XFKSYUOjD5x6iSVKCsl1bU7ElpHg5CWrt3Am1vCWRL1E0s597rpeNkg2nVi1/MqS6ioS9DQVE5ZdSyErgcBgkVd6TiqkiPCfWeyPtmMTzbTi1JqD1PJhUB7VMYbyAVpOjM7sMQSrQFl+MWX/vZbYIvlyPf8rLnlcKucDTYz7vRosevTj5qmBvM9tTbgOjy7/M3CLKDpp47k5Esn0N0WBlRKrL7d7YHBywb0tvsosUg45VTER2BnSuhI9+yKkJXsxTh4QZuA6sRo0l4XvvawHCW9HTnzzO/XlgKTg5z5FnAv0Zyew0US7AH3XoTTAq2xLUtOnDoJcrkBgVQS4dIXLl2+y5+fVUNXa5pcD4woOwLHiu3CrkdxqpTmmyssLOVgMBgTYO1Xel9jK4fyeB2tqc1oXyitismRJzeYlx7Y6IuixARcGsUJ1uHiGakB9P8oUXIUwJTRDWrS6Aa8nLdb/scAjmWR6urm+ZXhML2Sihhjp1bheT4VJTWUJ6pxbBfXieM6MVw7PB0Vw1YuIgptfMLeadlvc2aMpsSpwhI7uiaccGGjAMqECcULDmdwbl5XnKBEkkBw8pGTxSlJ4gV6wGmCtuvyxsZmNrSECPDRUyuoGJEgyBnidinaBNEYpIH/HajG4cLsUAyOFSNml4AYsmmfMdOqqKxPqHCSpRwVbSSqD5dUxUAyPy96IHP6zGmDQhm0ARybZ95YVWjkbjq2FtsRFDYxKzmsuZwHYnCHsqSwQ58gxKwkCOjAkCx3qG8si+YJUR81nXA4MiCI1PrcQGvirqOOn9KEGUr/a83C13fp/6Zj6vC8gJidwLacYQ3I3t9RZm7cIZfxiSV37VJiKzeSUrAcRXldPK9ebcJN3g47BuSxlI1KqSMBpo8bI40j68kNpP+jcTGd7Z28+OZaACpqE4yaXEEu45NwSxHUQWVAOInF4vavP8z1Z93Eg798nkTSRWuNpXZ1XokIyXK3OPNddTimo/M/TxIRFwjmTp8i1iCTTbQxWK7DkvWb2NLaDsCYIyspq42hfUPCKT2oS8wYg21bdLb28vBdL7JjSwcP/uoF/GiSoYjVZ+RmsmJABhyWNmBe5D2Y02ZOhcAM2t+LY/O311cWHqtpdh3KEixxcK1kNKvh4Ot/N+YgIjiuXRjGamH1meZVUu4c9hLgh6MU5JRAG8oScTVn0gRMbmAcvxIFns9Tkf5XIjTNqsHL+cScJJZyDlmcU9g0uuAsGEQUil39yIkKt7j8VnO4MSDPhMlKZCJgjh4/To2urwm7zwfS/47FjtZ2XlmzHoDKhiQNTRV4GZ+EU0p+8iJvyeg+UJLfMDrcyztZ5haXP2sOt1xQPvt5iohYgH/K9Ck2rkuQyiD90gGB1rhuglfXbmBnZzjxa9z0KkoqXXo6M8St0vwe8AeNCbuv+uLX8yOXFSaqJSRCFZR/f/XhxoD8U5yRr2YtOHYmooS46+xWgPGDAInHeGrpLrj51JNGYMeERDxBSaIUJQpzEPV/3gg7rt3HJXVcGyUaxOBmHYzysRwLJ2YRTr/CAJV76IU+5OnofHbwSWCugJ7RONaOOXa0x8ruO1IopVjdvI2OaCRMfWMZ8RIbtApzP4dom6ogCNi4qoXA18QSDmMn1hekwtOZsBncUvR2ZGlt7s0huBh+CHz0YE3A2p+E3EeK0Mbm7+zMP1NHBLWUwwXYaxW1nr5MOMpxhlLiKFGS36RnoLP/yGC1D9uVH4iz2Awo1fdvhPdqQHzCpsJrop+HDbD3/wBXAr5lJcH0RAAAAABJRU5ErkJggg==",
    "typologies": "iVBORw0KGgoAAAANSUhEUgAAAGAAAABgCAYAAADimHc4AAArvElEQVR42u2dd3gcxd34PzO7V3SnLtmS3IRccaG4YBMbMKYHDCH0BAiYEoLzEkrykoRAbCAhIQklEMD04lDzxgTTAgRTDAEb2xQ3cG9yU9dJ13Z3vr8/7k4+CcmWiSH4l8zzzKPT3u7czLe3mYX/tv+2f3NT6b95wA+A3h2u/7ftBHAWYGd1/QXG8aXHOh0Q4JdZ13e36Q5zsnaFSHsvBbwGvHTf2ffdaW4a8IH0Zzs9hpP+K90EPIDp4nsrPY7Z2xGg04vwiqAgaVkjRWSgQI6l1FZlWZ9EksnP0sDXOwEIWcDNB5qB1jQ84unrJUBdN+ZkZZBdGKTSdawDPZE+gOdTaq3J8Ra1tFDT8d69sVnpv8U+pX4LbEoDKrsntVKvW3BMB8rsSu6HgQ3AS8Cl6TFuBO4GmoDKXYxjpal4jK3UrDQSO86p1lbqAaCqwzr2KkWToeYxWvGUEQbk2hZHVBSaA4vCJmxr1rUm1NvbmqyljdHUKhW/94Sr0ws2nYgSnZbzs0khrDmLG/KBFcAhQG36fumM8jVcZuAWwLdPbpBJ5YXe4LyguCIsaYzqN7Y26u1xBwX1AhcAz2VzgtqLgD/MUuodT6TonKqezvSR+9gDSnMVWreRkokm5ck1282VH6yWmrhjW3C7B1d2wfoZERQEngWOS4ufILAkjZQtXeiBDPD/x8CdQUvLb0fu4108pJcVygvugKkI2xujcsuSjd7vlm6yNRgFx3vwSmaMvQUBtlbqfSMy8mfD+7i/GTfQBpi9tobnNtYRdT2GFYSYum8vSgpyWLq1SY56bbG7LZb0aTjZ60B1XSBhFvBNYHEaGZu70CMaEB8c4Cm1IKAVz00apo6u6qmTsSR3f7qZD+oiWEpxZEUh5w0qB5/NfZ9s8C6Zt8rSSm03IiOyOOtr3WyAoNbfA+SI8kJHzj9MZMpEuXvcwI6yVkYW50rNmd8QuWiSPD9pmAsYrdSSLFNzZxZMIK0HKjqT1Z+T+0o9B8hto/s7ctEkiZ49QU7qW/K5Of1keB/xzjtM5MJJctHAcgcQn1I3Zdanv+YIMABJkfMBuW6/vgqlaY453PDJBnIszSMTBvP25JEc26uID+tbeGZdDcYTJvcrtQ7tWSBGZHiubY9NA8TaiS/gAPekxY6vC/NTA15RkL6uyDGV4YB8f0iFZTzDe9ubmb2xjlHFucw59gBmTxpOWdDHLUs3sbopihjDz0f00TmWFlfkO8PAD7j6a27vm/5FFBiR/fqGAmpMaZ7GGCKOx9ZYkgF5Qc4bXMGhvYu5engfABKeQSOgFSf3LTaAuMaM78ToUFkK2ukgajL/6w4WkAJwHGs0EDy2V5EJBWylgepYEq3g5H4lTKos5cSBZZzYtwQBPBHwDFUFIT2qJFcJVG71+wd83f0ABUhLNFACiYJ+4QC5PhtE0GoHVDzXw026HFpWwJpvH0TvUIBk0sXv99E/Lwig0nZ5p/5EIEB/7ehjXJGRrkihVqrVr9RibZl/tDos7mC/K1IA7QUwMC8oAihJY1LAMYLnuiQSLreO6c91+/WjbzhAwjMEAj4G5AbNu9ubted55cByey9wewUQV6RNKmQ+ZZNzwjMsbmwlbFsUBVLL8nbQtNcJ8Et8Sv06kZBzwYTabhQhJgIGz1bqb67Iz4BVHcSXQArYdLBRVcr4QSnF9liCz5pj9Ar5UWlEeekblVKGLxg7+aqaAJj8RA3QsK41QWPSBbUD7K4IlmURyA/xyuYGvvXGMu5buQWfL+X5L2tK+QS2Ums7AH+IpdQ/HZFLynP8oSuG9Xb/dsQw971vHuC+dNQI97r9+7mD8nMsV+RUrXgfmJTlXWMrtQHg0+aYVrKDSlSaC+yAj5xQgP9dtI4T5ixlaWMUv23huR7Lm6Ia8MSyNn7dRZAAuqaGFq34ZFssedSiuhYzqU+xVRSwGZSfw/KmGD/9YDX9C0Lcs7wagAF5ORgUxnX5y/paDYiyrLcxJsMJPSylXvZEqs6u6uH8YewAu7wo18b1wDVgKb45sJyr9+vLjR+td363dFOJpdRsLXKwA8sA5fd7HxIj+ua2plBrwpGcgE8NzstBgEdXb6Ms5Kcl6fLq5gZyfRbFARsszSe1Td4nDa1aKZY3JRLr9gY/zAYIaD0FkG/2KnJkykQx5x0mLxw5XMK21c7kO6VfibSePUHk4iPk7rEDHEAspd7KHstWaiYg51T1TMqUiSIXHC5PTxwq43vkyz65QRlZnCt3jh0gcv5hIhdNkun798uM835WtBNbqecBmTFuoCMXTRLn3EPlqmG9P2eG/mnsQJHzJ4qcP1G+2bsoCYhf68szc1J7Saw+qJX62IgM+t2oKvd/R+5jg7C8NsJrmxuJe4ZB+Tl8u7IU/DYvrt5mTntruSSN0Vr4hgvzAXL9DG1JsrhvOCCLTxxl5YcC/GNDHcf8I6Vrc30WLU5KXTzwjUFM2bcXygjjX/rIe7+m2QpZ1uSo570IKBvGGaXey/NZ7qtHjbDG9ipWuB6vbarjw/pWLAWHlhUwtlcReIafzFvl3rKs2raUWuqJHAQkALH2kiBcUmChVurcV7c0+La3JpyBuUG1b0ke4/qWqAm9ixlaFKa6OWZu/ni998P5qy3HiFZwmZcKMwQAV4v+vity5NXD+pijK0u1MsIvP1zL4sYod48byCOThvON4jBPrathdUucSwdVoC2LAq3MXzbUKgFjUuP5DawHdNwzk/6yrtYUWMpU5QbVfuWFakLfEsb3LqZ3jl/mb22UH7y70n10zXafpVStJzI57WUrQPaGcHRG+f3TiJyoFI/cs2JLxcOrt3FAUZh+4YCxtaY6mtAf1rfoiONpoFnDlQYeSiPQBUgYMxrg8PICJZ5BKUWz46GAw8sKyLc0J/YtYUxJHv1zgyitEM9jbI98nWNpFfPM/pIyAxzAEpimgCbH/eXUeauY/vF6RhSFKQv6jGuElZG4/qi+RQFaK/WxJ3IusDQ7xLG35ANMGpCvijDap9QVcc+cPK820n9ebSR7Det9Sj2vRW5PwOos+12ngz4lCugR9KFStmIqsAO0eh7iCY5n+OCk0SlvzHGxlaLIb1Hot4nFkgVjRmOzcIejJjDNhldQ6kfb487EOVsay7OsyySwxK/1Y0lj7k0H+9rFpPamhIyXnvwWR+Sno+Ha9X5/lWtMuYhYWuuasOOs3iQS6yT5odLmd0yAVte0mbMZ+92nNEYErTU3LFpLWY6Pi4f0AiPEPUM05VQkJi/EW9iBMFz4JyL/LIIC4/NVYUyxUsoxlrU5kkisThrTLpSxN2fEMt6otRAckskV6bg9eB4N7dN/Xkdl7ldqVVLkmOVNUTmwZz5KQCuFpRSNSRcrHGDdtiamfbyefuEgFw+qAEuzojkmTUlXNKy+fgc3eh1EpGqAJhznox2OipsNZ6+zDJ1m72uSlumqg6LW6WudLVQALK3nAOov62uVSt90dEUhngjnvfsZZ7+2mBPnLAXglH4liFYorXhw1TYBVEDrl7pIZJks4siYqlYWfN1u5pbZ20pJTk33XSWXVLrnakW1rZQ399j9PblwkrR8d4JcNLC8ne1+fO9i2X7GwSIXHyGvHjXCs5XytFLbgeKssfboQvZG4JdAW7K7BzsS6LKz5LlPc5FjuL9fOJB88agRvhE9ChQirKyLsDWWpDjgY3hpHliaOetrzOlvfeo1JByfgnMN/HlPJ9atvRAB/jTLH68Up6d16XxgeVrWmp2ILssICxVUNjnemCfXbMd4xiuyLQbl5VBVEKLUb8mi2oi56cP15vIP1lhR17MU/Ergj3t7VQN7KD2ZjiYyNw1soxTvdFOvZWQ0luLXbd6oUtInFJAh+TnSK+TPFkc1Gi7ei4n1SxE9+yjFY4D06mF5vXpYHiBK8Sg7ykhUFzpAZSPBB/v7lPo9MC/tnTaQKneZ69f6mpwdZYpWJ2Oo/0TKv1UpmrVCykst790He8k7D/aSnsWWpxWiFM3ArR2eUd3B7MRKgv0KCoqOG0jgq9Sh9l5kegKMVIo8ESgttNSQffy4nlCUp1VNg4eCPIGRHZ4RIAfwUVIiJZTusB2No41xtWX5vRXJAtdfmBv7uLXFKi5uyve8pGVZfk8pq4NSr6Wuri4zbst/ihXUFhUFTtOa64xh0Pj9g6I18s5Hca01y4zheuD5LJffBU4C7syq+VS7+A3ppkMIMBP4+b+inPc2DogBM43hFaX45z8/ifdPZfdYbwwTO9Ta7OAApfohQqioB4HcfDwngVJ6Nycgbc80bV5PFkH8R/kBKm2GJoALtea+lCjhSuCOdNg5mQX8VPWZZd0onndtj4Ejkmfd9bzP8gfEuI7KTm/uEgHGI6eghJd//UN38eyZPqWtt8V4R2aVPcp/iiOWCWD2A9alr/VPf1Yd/IC2UnWl7RfFuMfve8xp7sm/mWnHI41orXcNAgWe6xAu7skHT/zJvHbzlUppvVWMGbOT6rn/rx2xTP67CSgA3gOe2Yn8Tsl9MS9ryz6tZtWSUqW1GTDhWJVobUZpDWJSpQwdO2Ach2BeIes/eFOev/YCAUS0PhFjlmTVFe11Iqgzhac6+b+7uqF7Nfw+3yhtzLtijO+UW57RgyaeoGJN9WjL7kLsGOxAgGhDHTMvmORGtlXb2rZ/ZFw3o9TdPeZZ8tWlF3W2zBQRJSIZ4Kc+T5umO6mz76z7u2lIeICN4yxC9CUC1ks3/MCrX78Sf04Y4zqIMe2754FSKGXxwi8vdCPbqm3Lth/ek8D/VzlAZVGtdLfSDaCysjIYjYbt2rpPWySdrFBaI1msP/r73/dtePbZgNYVnY69bfvi1sy9uyGHbcBVlu928ZzLy4eOdM++/1XbDobaAJ5N/f5QLn+/6X+8hU/dY2nLt8B4ziFpwJs9FV5W/8JzsrviRmv9PRGZIiL7pE24xaf/7pnv1W3ZaM257cfPAfNuapJf/m5I3+GNWzc9QGoHi3QyTwHqlGW9Ip73u7T52Z05ZZSyaNv3unGdw4cdd6Y39rtTLScaSemDNPB9OWHWL3zPvHHHNVppq06MdxCw9l9VunsyLBBOOz0PZ+0M1Du5f0aGW/J69pL8sj6yz9gj5Io51U2Xvba+uXzoSDnwlAvlF5849d+Z8XKiuHKQ5Jf3SfWyDr28r2jbJ6kYkFqRjgGpborUzD3lSuk1WcmUzkSch1IxUvsFvhS/SX1Byg+T2tpzBPA34Ns7U35a63OMMTPzevZ2jvvFn3TF8NFKxCh/KFcUSqMUxnhGKa0QUUpbJOOtxjiOQiFKqbS10sYAKtbUIG/fNd35bM7fAkrrV8WYY3eDOn2AoxS3oPRVBw0Luv3KfHbSlRQWNbTGjTfng6jlumYBcNCXRfn2F5D5+aR2kxyRZsk7gTFpYDcCn3W0UkS4VClljrvmTj3kyJOt1rptKK0xrqtEUoJcKaVFXERElNaEC0u18Vy0ZSvPSeDEYztEhAhFffqryTfcH6hdu9zUrf3sGF84PMJpbV3STUAJoGyNdjzDz84r4dsnFJBsclFa4fNrNm5Jsu+3V+G6yLRp6Ouv/3JSivZuWjAu8Ps08BOkdv693uG+8Wnb3AbcysrK4Pr16/uFS8p0xYgxprVuG4hBPElbGTs0nwCWz6c8x2HujBvY9NF75PXsxeizptJj4AiSsZY25ykeaSRc3JN+YyZK3drPUEYNJrW3S+1meIPmVo9YvUtDxENpRdBnqG/02m644QbMl2Wy218gHvMycF7aBNxKasNZpry7Kc0Vbbtb4vECC1CWz58Ky2jdBvxOBbRl8/y1F7Dizefbrq2a+zJnP/APCnvvg5uMs0MkCdqy2N3wc8dWmGeRU2oT8KWS8CqgiCXNV+Ik2V+gQu1Z4GzgyTTg/5buHRWdAizXTWTkRjv/y3jtpYRS4A/lsXXZQla8+TyDh+zLzJmPcf+DD/HAvTNY+uLjHH75TTjxKMrSOx5q25Ao2dUI0h1T0U1P4dHnG1mwLEYsISgFtqVoingkXXZM/WuAgAxV28D/pT//FbgtbQ1le7dtodn6+pWRdoAQATQ5BQXttjXEm+sBiDfVgVKMHj2asQcdRH19Aw/efx/R+poOSOygoLSOdnJ8wU5N0wxgn30zwrNvRvi3lX/vZnPTVDaL1EbmaNZ+qgzlDcjLyytL2f7+nKamukDavlbKshFxWPDEnWxZthClFBXDxjBi8tkEwnkEcgtABNtOiZY33ngDMYZgfmGa4qUjFFWK5eSA4uLiGmOMP5FIxGOx2MfpyKjVxSZtbSmUJ3DZmcVMOCCHlliqXtRnQ22jx8/u3E7SSXHFl8UF9hf0AzIb3N7tJEnxQ+CWSCQS6JRQLYvnr53Cijdmt11c8tJTrHz7RY752W2sfPsllLZYs2Ytf7zjDv7wu5uxfH6GHH0qbiLWWRxfA8SjLb+KR/lV1vWFwNGQKZhr15KkzM2k58GkMSG+PbkQN20FWX5N9ZYk1961PXXj14gDMpkfo4C+BRTF/ASUSlGYz83xVdfHrguFQoGjjz7asyxLu67HK39/WXmuSyCcS/XH77Hijdnst/8B/PrXv8YzHjfecCOL5r/Bg2cchOc6ALz7zlzefWcu2vZz7DV3UD50FPHmBpS2YEetJRkj6uBvjJe+fVI59BUrVngff/zxaJ/Pujrk854WwacUXlMUfD7IybEt5Son6Tm9XQOei0o0uDRGPLRWBPyGhoj3lZSy2bvpsHl+P/uKqCmOI0dsaKJv+6xQXAO5FRUV8vTTT1uBQIBIpIVB/fehIZlE2z5i9alaqlGjR3Pi5BMAmHjooex3wAFUb9zIkEknsf/J57P9s4+Z9+iteK5LrxFj8eeEU06S7cNNxEi0pmS2TvsGP/7Jj9Vpp5yScrlnzLAvvfRS4zj8rMnhZ9kLcZxU1XN2m7csZp12XD5W1KCVwrYUllZfGx2QrVynJ5NcDZJjaUWfMptwjs5sL8QYxYpNCTzP0NDQQDAYpLk5goigtIUTj1I2bDS+YA6PPfwQG9evJy8vF9cz1GzbRumAYZz460ewfH6GHvNtPNfhnftuYvY151FSNQTPSZBXWsF+J59PadW+SNYuxZaWFmKxGFprYrEogB7azy/9y5USW0ssKbyxoJXCPM2EA0J4JmVMzfmglT/MrKMoz+KaC0upqXeRrzBKb3c3eGVZPIFwVs9imyknFbqnHpGn+5X7lM9SiAaaktTXuhxw6UblGrBtC8tK9Yyo8JIJCnrvw/HT7uMft1zNnDntfbjCXpX4Q7k0bV6P1n0o7T8MgO0rF7N95eK2+5b+/RnOvGs2/UaPJxNN1VpjWRZaa3xpBf7dw8Pq2rPzMfl+taHOo2rySgb38/PCHf1IxIVAjuL5NyKcdU01v7h7O0rBz6eUEmn1vlTTc3cQkErnKW7yPM4CkkOrAr5fX9bTdhJCS9TsMOdk15aC0ppka4QhR36LPgd+g+bqlM+WjLXywvTvs3HRO6yfP4d+B00i1ljP4tmPopTmpt/8hklHTCIWjfLsc7O547ZbWfT0PfQffyiykx9NOEJrqyGpDZFW02b7t7QY4kmhqVU48bA8nvpNb876eTXX3LUdY4RfXNoT2eJ8taV+XQHf52OUCD8vKbDcffcJ+OZ80KomX7aBeFKwLHDN7pGK0ppkSzPBvAIq9htH+YiDGHL0t6gYPoZEa4THf3ACT116PA99dxyr//kaIobly5dxwH77cfjEiUw+/ni0ZdNaX4PsIuKjFFg61bNOtWm75rcVNQ0uJx6aQkIoqLn2nhpueaiWkiILz/x7EdBmafcoshp/96Oe+s37KxlaFeDFd1o497pqLK3wWQrHEVxPuj1hpS2M6xJvaUJpi3mP/JFVc/9OXn4Bvcp6svb912nesoHBk06ipGoIjz36KCOGD+eEE07gO2edifFc+o8/ZtcTTx3R0NY7u6a1YmudywkTcnn8V70J+BU/uX0bP7x5qxiDAbanmUx/pSJIBFEK/Lb+cPXfBtb7fbpQEPPiH/uqk67cyHNvRfjutdU8+ave9CywEe2hkrJbkTBfIIfItk28eed1BPw2s2b9Fcf1OOH44xl29Cmcevsz1Kz8lH/8/kpWzv07q9esAdvPqDN+wP7fOo9ESwKtu64rCPoV4VxNKE/TnExB0bYgN08TTEpbJEMBSVc4eXIBL+QoTvvpJpk1J5Lx+p/49+iA6SilkMmHhidvrfOqykuReEJ0r1Ifz9/Wl8lXbmT2WxFOuGID39g/ZGLNrjgJIe5iFSlwXQ/b9vA8r0sMW/4AzZvXkWiNMHnyZI468kief+FFxHhYPj/J1ig5hcV86+YnaNiwCifWQjC/mKJ+A0m0NKXQqDJlhgbP8xARHDf1m0+91SoLV0YxtkUsjYCVGxxOuWojnulkm0vqFmVSFpLfCLNEeGo3T2HcIwjQ6nrMoD45vZ9+NfJ8XaPhlbv6SdIxRKIeFaU2s2/tywmXb+DNhVHeXBhtx55+W1NcXIzf78eybFSnkU9pF49JOiml1xptbcsPK23hOgm0MRRXDkqJLs8l0dKMSPsIcW5uLjk5OQCEQmEAWbIuqZasaw+3hojHs2/sMu6zBXgEmPZlby3qCgFm2jT0vMdjtaaP77ZLTy+60nFElEJZWtHYYuhfGWDciJCs2JBUeTnWTa4nS7S2ciJR546m5ubAE088IT6fT8VicWLxuK0Dee2WopTGTcYprhxETkExr73yCqefcSYLFy5AjKHPgeMRMShloZTCScTSaRSVDmmbtuQMwNy353rGdUUpxfvvv+8BAb9f/ylgeX8Wy7aV57qRGMq20Tk5Nl1ES5X23ERTlDXp0Pru5r/3XErymWewzjgDT+T03ixbtqluiyuWlbrf9YSSApuzr602T77SpIvz7XH1ze789KN/Toer27X8sj6c/+d3sYM5bRUIYjz84XxWvf0if7/pR0TrtwOw/7fO4+irb8G4XVd+GM8lVFjKqzdfwcKnZ3QVPt+f1AEbfMESGvPv4gCWLkUmTsQePmD21D9cXsaho0ISjRulFWmLZ8e8XE8KspLylwDzfT5fv5T/ZQWTyfj3QHKN54nxXLWjBEQRb25kwITjOPehOdSvWUawsISyfUfiJuJZYkY6QYCH8Vwk5Q4rXyD4Esb7NCXDTdLzvBfSwLczJ6+kT8L6LdCL1DnRTicn3kon21y/cgTo66/H9Minf01z4prfPlLLN4/pr0IRg7bAcwWr2CbgV23lk1lh6iRwh5OW6Up5ACcobeXmFJWIL5CjOtbgGONRWrUv5fsOx3jgxGLYwRzUTmwqz3MIFYWxgyEDWJYv+Nt4S+PcTjjczQL+74GfkDqsNfF12O+1U0+4KY6tFGbTdlf/5k/bVSY2bgTCQc3HK+KpEJzrdVy0nV7wML8/cJjnOfleIsYnf31AaZ8/Fc3soJhFDGIknW7c9Q4gMR7+UC51Kxcr27YFkzyHVLXGa9lHkmV9/gPwY6A+HTLvk55njFSRLV/HLUFDdlEamKmnOaKTDdPj0jtI5CvuN2XNJUNgU9PfJbI2TWf30/5dxco75YA8P0SSMLifnx+fU0IimeEAITdkcd+sBuYtiREIWCQSXjbyXKXUd0UkfMYZZ8SHDx8ejMfjWLadJdPVHimSNiZ12onruu4999yjm5ubL6ysrLxh/fr18SyArkzvmgmkz3eel/6s0iJz3W4W+341HFCSl+KAI8aERZaOEG/+MJEFw8SZN1RkxX5y/omFHiAFYeuIjlRnWdafAPPiiy86IiKu68qX0YwxIiISj8dNVVWVANUDBw7M77gtFZicRkIMOJev05FgnRTbpmvq0+dkukK0waUlatA6ZQWVAomkdLUNNEPeqrGxkVgsRmNjY1toek82SR/cF4/HM1636gB8k66Ee4HUSxpmA4+RKiSIZK3ZdFFs/EXK5rtTXSgdEZD9w16KilMp0Uyphm3tOCvbtnbs8FFKJbMO0WiXIw4Gg+Tk5OA4zpeKgEwuAJCBAwcmVq1alW1KZszN54EJpM58aNxZsda0VIk8119/vUmXz6t0Larsof1ubRyQwYi/uLi4h9baGGO01tok4429u1MGL+jy0tLSCmOMJSLKsiy3sbExX2tNXV0d1dXVNDU1fakISCQSGGPQWlsLFizoV1pa2iIiSiklmb/GGKu4uHhJKBTyNm/eXNHFeKqurm7z9ddfbwD69etXpJRqyACuR48e5cYY/UURYYzR9fX1NWndozIskTmyPZZmy9Z0jwFy2MiQxN4dKjX/GCJ1rw+Rba8NFm/+MPnOsQUZrolnPZPpToartNZfSc8SJdFO5pPp8S7m25q19ofee++9QYMHDz4TqC0uLr580aJFg3w+32/Sll1kJ+PvrEfSMF2chrmydyTb/Ut79+49XCmFpE+O8twk69ZvRGsIhnQ69ZfSATqk8aUPXezbp1cgEAzhed7nAm/GmJ1mrfao5aB1ptY0JztOlF09sfMETmrthx9++JSWlpbzpk6dqh944AFOO+202xsaGm697LLL9HPPPde2ru6Om5lLZvzq6uqlyWQypbCmTZum03KuePbs2Zu01kHP8yQQCKi1a9cydepUVZhnMXJIENeTtqN5fT7F8rUOW2qT/OpXN8rYseNobW1tq1JoYy/11SW4RQQRwRgjWusMQhCRtrB41vy6bD6fT+LxuLZtG621iAjJZFL5fD5Ji7l242YIdhdrFa21MsbETzrppD5Kqfpp06bpNiX81FNP5fv9fmVZlnJdF9u2lZv2cBsjHm8saO0ylud6Rrmu2zaZf1fzPA+fz0c4HFau6xKNRvFcF9vnIxQKYVkW0WgU13W71EciguM4SmstjuMoQHmeh9/vF7/fr4zntY1h2zahcBjLsojFYiSTyZ2Oa1mW8jxPPfXUU/lpj3yHFRQOh00mftMWDrRSAB4zLMhj1/emNWZQKpXKK8qz+MkftzP7rWZsrb5SSu9CuZGXl0ddbS2vvPwS8z9YwNqNm4gnE4RzQgyo7Me4cWOZcMih5OXl0dLS0imwlGpbi1JK4Xke+fn5NDY2qtdfe5V58+ezZsMmovEYoWAOVX37MPaggzjksMMoKioiEons0tgIh8OmW55wBqbhHM3QgQGirTv8gNxCm8I8q2OR8r8N+Ll5ebz+2ms8/Mgj1NamTizI05Dvs2huauL9rVt4f948Zs16losvvphxBx9MJBLZqUgyxpCXn887b7/Ngw8+yJatW1PJHwX5fouW5iY+2LaVDxYs4K+zZnHBhRcyceLEXY6725VxxkAiJkRjkkKAEUIxwXWljb2MMW2KqTvKbk9xjDGG3Nxcnnz8cR597DEApgzuxfkDyhhdHCZsa5ocj/drW7h/5Rb+unYTv5w2jSuuuILjjjuOlpaWToGVQerfnp3FjBn3AvCdAeVcOLCMsSW55PksIo7H/PpWHl61lcdXbeWmm26irraWU049tdtIsLtvYWT1tK+cGT8YCBAOh1MbJrTeUXSuPl+loJTCdV2SyWSXSDBZxVbdAf4bb8zh0cceIz/g48+HDOHEqh4ArK1roTHp0iPo49jKEo7tW8TDnxZxyXsrufOOO+hVUcGI/fYjGo22+y1jDOFwmHnvv8+MGfeSY1s8PGEIZw4qA6XYUNfCqqYoxQEfR/Yt5sg+RZzUu5gp767g3vvuo6JXL8aN+5xRsudOSzEGdCBVjqIVPPTwwzzzf7O6TsBnWUOWZXHqqacybNgwEolEp0jIzc1FKUVLS0uXpp6IYNs2jQ0NPPjAA2ilmDlhMCf270ljS5xL3lvJM+tqMu8P4KJB5dw+dgBThvUm7hmmvr+S++69l1tuu+1zQNJaE41Guf/eFOXfe/AgzhxcTmtrgh/OW8Vjq7e1Edn3BpRx17iBnDG4HMcI58xdzv333ct+I0ZgWdYujRJ7V6Jih3mXonrHFXqW2Pz5b4288E4EI7Bs+We7hcCysjJGjhxJLBZrp7CMMfj9fmbNmkVraytnn3028Xi8zYbuTOm+/tpr1NbV853+ZZxUWYpxPH67eAPPrKthYF4OI4rDzNvezIwVWzigKMzFw3pz6ZAKnlxbw9w1a/ho0SLGHXwwLWlqzXDVm3PmsGnzZr7Zp4RzB/bEczzuXF7No6u30S8cYFRpHotqIzy2ehvDCnL4yf6VnD2gJ0+u286LG7cwf977HHHU0TQ3N+9UKeuuAG+MIZQTwrJzqN6eJBoXjCf0KLZ5/KUmply/mdaY4feXl7Fg5gDeuq+KhY8P5JHpfdBac/iYXD56YiBvP9Cft++v4uMnB/Krqb3QWhPw+9pRRgbAmZjOyy+/zOOPP86MGTMIhUJtNndnc/xgwQJQiosGVaSIxPP4e3UDtlLMPmIYzx4/kicOHQLA/22oxZLUC34uGlyBUopFCxeisyg1M5cFCxaglOL7gysQpbCM8PLmBiyl+OvE1LjPHp6qXX1mXWpcUXDxoIr08wu75ah1yQGO41BUXELVPr1YvXo1S9ckmDA+zMy/NHDB9ZtxPeFPV5fzw+8UE4+a1Llg4dRrOowxFOTCASMCxCKpk0hyCiyWrE6k5Xt74Pt8PpLJZFtE80c/+hE333wzs2fPxhjD1KlTicVinxMTsViMDdWb8YswND+IMqbtrUUGKPLbeC1xxpXmc8ngCiaWFWAEtAgHFoUQEdZv2oTrum1iSGtNMplkw6ZqEGFEYQjlpTJ4ngieCCUBG681zoiiMP8zpBcHFIdTIWTPsF9hCNLj7kzP7RIBxhiCwSCHHjKBVatW88BzTWypd/jeL1OvCvnT1eX88KxiampdLK1SekEZoglpOzbZjZpUAa8Ifp/KDl+3AT8QCLBq1SpuvPFGEolEG0JisRi2bfPCCy+Qm5vLlClT2lkWSqnU24qSScIaAlZmI/eOTLJrBA34tWbGISkuSCZc/LYmlBYLsXgck6W7lFIYzyOWiGMBIUt/Lg6dNAaNwiDcechgEHASLj5LEbJ06pWsiQSmk9BMt2tDM4ro6GOOo6ioiEefr+eMq6sJBTV3/bSCS08rYluNi9aqXW4vO9jaWb6w0/oPyyIQCLT1YDDYTjHatv05EZRBVDiUQ7OBiJMudUlTP0COpVGWptXz2P/ZD/jlgjX4fBYCNKS3QObn5mJljS8iWLZNXjiMm7lPpd7Skxk3ZFsoS2EERj+7gCveW4HPl7L+GhwPB8jLze103t1GQMZcLC4u5rLLLkufYSQcPiaXH5xWhFJQUmiRH9ZtPZir2zZs+GyFL3fHd3auJiegPvcbiUSCyspK7rrrLu6++24eeughrrrqKkKhEK7rcsopp3DOOed8zqQzxhDMCVLVty+eUixpiiJaobUibKcKmObXtRBXir+sq2FxQysf1rekskxaMXd7MwCDBvRvB6iMIVBV2Q+lFB83pMZFQa5loRW8XxMhDszeWMei+hYW1LWkqiQtzXs1EZRSDOpfhc/v36VvZO8quhiJRJgwYQJXXXUVd955Jy+908yIM1bz3eMK2X9QkFBQt5mmoRzF0tUJtNbUNQv/nB8jEk3Rfn5Y8+m6TAxGtUOC4zgopfD7/di2ze23305dXR2nn346F110ES0tLZ9jZUmbZaNHj+KtuXN5ZNU2TuhbjFGKCweWM792Jce/voSyHD/bYqmtdj8YXIEoTSzhMOOzVCHEmLHj2snqjHIfOXIUL778dx5ZtZXv9i/FA6YMKuPVLQ2c8fZyyj5Y3TbuD4f0wmiFcQ13flqNiDDu4IM7jQ7vth9gWRaRSIRjjz2WPn36MHPmTD788EOuu2fbTp+bu6iFCRd0fqxm0nHbTSzz2fM8bNvmlFNOIRqNcvrpp9Pa2tolccRiMcYe/A1KS2Yya30NL23oyfFVPbhgUDkR1+XBlduoTTgcUBTm8qG9OaGyFBT85MN1rGyOMv7ggxk6bFg77sqMO2rMGPr06sVrmzfz1JoazhpSwRn79KAh4XL3Z1vYFk8yojDEJYMr+M6AMtCaX3y8jo/qIhyw/34cOHLU5xy8L+yIaa1pbm5myJAh3HjjjaxY8SlLFi9m06ZqorF4uySpIGmPl/aFVem8m9KaiRMnkkgkOnWAHMdh8uTJbY5YV2GLDOcUFRVx3nnnc8utt3LOuyt4DJhcWcqPR/fnxyP64bkelt8Gv01jc4xrFq3jnk+rKS4q4uJLLmnjvo5R1dzcXC648EJuuPFGLnpvJVopzujfg0tHVnLpsN54joflsyDgozUSZ/qCtfxhyQbywmEunTq1LTT+L3NANifEYjGUUgwZMpQRI/Zv+5HdbY7jdOkFk95w151QRIY7jzrmGKqrq3nq6ac58fUlfKuyByf1LWFoQYh8W7Mt7vD29mYeXbmFdS1xepSW8Itrr6OsrKxTKtVa09LSwvgJE7hgyhQeevhhznxrGY+sLuHkylKGF4Qo9FnUJFzerWnmsZVbWNEcoyA/n59fcw2Vlft0Kwyx26GIzICxWIxoNPqFA2q7CsZ1N5LYZq21tnLelCn06duXJx5/nOfWb+G59TWdhnePOPxwzj3vPMrKymhtbe3SS80g4Ywzz6SiooKZM2fy8saNvLyprtP7Dxk/nu+dfz59+/btMtS9x2JBuwOgr6IppWhtaeGoo47ioIMOYvHHH7F82XI2b91KPJGgMD+f/lVVjBw1igGDBuE4DtFodJdAyiDh0MMOY+TIkSz+5BOWLVtK9ZYtxGJx8vPyqNqnkgNHjmLwkCF4nrdTpHYHAenz+JXsbYe6Ziw2v9/PIYdN5LDDJ7VLF2Y83IxS7y4RZca1bZuDx49nwqGHdjpuNBrd6bhZMDWdIkBrrZRSIb/f3y3z6evcksnk5xRgBmDB4Bc/7tlxnM+FF7o7blYBWUjrHdvw7SyFlhSRTxKJRCBd9fcf9YKCr0BMips6oi1hWVay0zI7EVHTp09X06dPl+nTp/8XAXuwZWA6ffr0PVFd99+2p9r/AyqnJ+aO8APhAAAAAElFTkSuQmCC",
}
print({k: len(v) for k, v in ICON_B64.items()})

In [ ]:
import base64, json, time
from sempy.fabric import FabricRestClient

client = FabricRestClient()
WS = fabric.get_workspace_id()
RPT = "Fraud Copilot — Dashboard"
rid = fabric.list_reports().set_index("Name").loc[RPT, "Id"]


def lro(resp, what):
    """Fabric long-running operation: 200 = done, 202 = poll the Location header."""
    if resp.status_code in (200, 201):
        return resp.json() if resp.text else {}
    if resp.status_code != 202:
        raise RuntimeError(f"{what}: {resp.status_code} {resp.text[:800]}")
    loc = resp.headers["Location"].split("api.fabric.microsoft.com/")[-1]
    for _ in range(90):
        time.sleep(2)
        st = client.get(loc).json()
        if st.get("status") == "Succeeded":
            if what != "getDefinition":     # updateDefinition has no result (GET /result -> 400 OperationHasNoResult)
                return {}
            r = client.get(loc + "/result")
            return r.json() if r.status_code == 200 and r.text else {}
        if st.get("status") == "Failed":
            raise RuntimeError(f"{what} failed: {st}")
    raise TimeoutError(what)


cur = lro(client.post(f"v1/workspaces/{WS}/reports/{rid}/getDefinition"), "getDefinition")
parts = {p["path"]: p for p in cur["definition"]["parts"]}
print("current parts:", len(parts), list(parts)[:4])
if "report.json" in parts:          # legacy format: keep its shared theme package
    old = json.loads(base64.b64decode(parts["report.json"]["payload"]))
    keep = [p for p in old.get("resourcePackages", []) if p.get("resourcePackage", {}).get("name") != "RegisteredResources"]
else:                               # PBIR format (report saved from the web editor): rebuild it, keeping only the dataset link
    keep = []
    parts = {k: v for k, v in parts.items() if k in ("definition.pbir", ".platform")}
REPORT_JSON["resourcePackages"] = keep + REPORT_JSON["resourcePackages"]
parts["report.json"] = {"path": "report.json", "payloadType": "InlineBase64",
                        "payload": base64.b64encode(json.dumps(REPORT_JSON).encode("utf-8")).decode()}
for icon, fname in ICON_FILE.items():
    path = "StaticResources/RegisteredResources/" + fname
    parts[path] = {"path": path, "payload": ICON_B64[icon], "payloadType": "InlineBase64"}
lro(client.post(f"v1/workspaces/{WS}/reports/{rid}/updateDefinition",
                json={"definition": {"parts": list(parts.values())}}), "updateDefinition")
print("report updated: icon navigation + FR/EN pages")

In [ ]:
# free the capacity when run by hand (a scheduled/API job ends its own session; stop() would mark it failed)
if str(notebookutils.runtime.context.get("isForInteractive")).lower() == "true":
    mssparkutils.session.stop()